# 3D reporter timelapse — 05_preliminary_reporter_quantification

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Preliminary Reporter Quantification

This notebook reviews the first aggregate FOXF1 (RFP) and BMP4 (YFP) measurements using the current ilastik organoid masks and the masked illumination correction.

The goal is to get to preliminary organoid-level traces now, while keeping the pipeline easy to rerun if the masks change later.


## Interpretation Notes

This is a preliminary analysis stage.

- frames excluded by the phase-artifact QC are not quantified
- denominator-based metrics are suppressed on persistent border-touch frames
- positive-signal metrics remain available on border-touch frames
- reporter images are illumination-corrected before whole off-cyst background subtraction
- thresholds are defined once per reporter from pooled early corrected within-cyst pixels across positions
- whole off-cyst background is remeasured every frame
- reporter-positive masks are cleaned into contiguous regions after thresholding

The most important outputs to inspect are the reporter overlay previews and the trace figures, not the tables.


## Upstream Signal Preprocessing

This notebook establishes the thresholded reporter metrics used downstream in notebooks `05b`, `05c`, and `05d`.

The upstream preprocessing chain is:

1. apply reporter-specific illumination correction to the raw `FOXF1-RFP` and `BMP4-YFP` images using the shared channel-wide illumination fields estimated earlier in the pipeline
2. for each image/frame separately, measure the whole off-cyst pixel median **after** illumination correction and subtract that per-image scalar background
3. after that subtraction, estimate the reporter-negative within-cyst baseline and sigma from early corrected cyst pixels, then call positive pixels at the chosen `N sigma` threshold

So the thresholded areas, positive fractions, and positive-region intensity summaries in these notebooks all sit on top of:

- illumination correction
- per-image off-cyst median background subtraction
- within-cyst baseline / sigma estimation for thresholding

By default in the manuscript-facing reporter notebooks, the thresholded positive-fraction comparisons use:

- `FOXF1-RFP`: `4 sigma`
- `BMP4-YFP`: `3 sigma`


## Setup

This section loads the quantification outputs, project paths, and helper functions used for image review and plotting.


### Load Analysis Packages


In [ ]:
import json
import math
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter
from skimage import morphology

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


### Debug Switches

These toggles make it possible to turn notebook 05 into a reporter segmentation debugger without changing the core analysis code.


In [ ]:
DEBUG_SHOW_ALL_RFP_THRESHOLD_EXAMPLES = os.environ.get(
    "NOTEBOOK05_DEBUG_SHOW_ALL_RFP_THRESHOLD_EXAMPLES", "0"
) == "1"
DEBUG_SHOW_ALL_YFP_THRESHOLD_EXAMPLES = os.environ.get(
    "NOTEBOOK05_DEBUG_SHOW_ALL_YFP_THRESHOLD_EXAMPLES", "0"
) == "1"
DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = os.environ.get(
    "NOTEBOOK05_DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING", "0"
) == "1"
DEBUG_SKIP_ANNULUS_QC_PLOTTING = os.environ.get(
    "NOTEBOOK05_DEBUG_SKIP_ANNULUS_QC_PLOTTING", "0"
) == "1"
DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE = int(
    os.environ.get("NOTEBOOK05_DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE", "8")
)
DEBUG_INTEGRATED_REVIEW_TIMEPOINT_COUNT = int(
    os.environ.get("NOTEBOOK05_DEBUG_INTEGRATED_REVIEW_TIMEPOINT_COUNT", "6")
)

display(
    Markdown(
        f'''
        **Active notebook 05 debug switches**

        - `DEBUG_SHOW_ALL_RFP_THRESHOLD_EXAMPLES = {DEBUG_SHOW_ALL_RFP_THRESHOLD_EXAMPLES}`
        - `DEBUG_SHOW_ALL_YFP_THRESHOLD_EXAMPLES = {DEBUG_SHOW_ALL_YFP_THRESHOLD_EXAMPLES}`
        - `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = {DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING}`
        - `DEBUG_SKIP_ANNULUS_QC_PLOTTING = {DEBUG_SKIP_ANNULUS_QC_PLOTTING}`
        - `DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE = {DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE}`
        - `DEBUG_INTEGRATED_REVIEW_TIMEPOINT_COUNT = {DEBUG_INTEGRATED_REVIEW_TIMEPOINT_COUNT}`
        '''
    )
)


### Load Quantification Outputs


In [ ]:
import sys

cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    TIME_DISPLAY_OFFSET_HOURS,
    display_time_hours as _display_time_hours,
    format_display_hours as _format_display_hours,
    format_display_hours_from_index as _format_display_hours_from_index,
    offset_time_hours_df as _offset_time_hours_df,
    set_display_time_axis as _set_display_time_axis,
)
from notebook_invariant_helpers import (
    assert_no_excluded_keys_in_analysis,
)

DATASET_DIR = ROOT / "data/raw/20260128_BMP4-reporter_LPM-organoids/d2-d5"
MASK_ROOT = ROOT / "results/ilastik/organoid_masks/full_dataset_v1"
FRAME_METRICS_PATH = ROOT / "results/tables/05_reporter_metrics_by_frame.tsv"
THRESHOLD_PATH = ROOT / "results/tables/05_reporter_thresholds.tsv"
GLOBAL_THRESHOLD_PATH = ROOT / "results/tables/05_global_reporter_thresholds.tsv"
POSITION_SUMMARY_PATH = ROOT / "results/tables/05_position_reporter_summary.tsv"
ONSET_SUMMARY_PATH = ROOT / "results/tables/05_preliminary_onset_summary.tsv"
PARAMETERS_PATH = ROOT / "results/tables/05_reporter_quantification_parameters.json"
BACKGROUND_TRACE_PATH = ROOT / "results/tables/04b_background_trace_by_frame.tsv"
BACKGROUND_OUTLIER_PATH = ROOT / "results/tables/04b_background_stability_outliers.tsv"
BACKGROUND_EXCLUSION_WINDOW_PATH = ROOT / "results/tables/04b_background_exclusion_windows.tsv"

PREVIEW_DIR = ROOT / "results/previews/05_preliminary_reporter_quantification"
FIGURE_DIR = ROOT / "results/figures/05"
TABLE_DIR = ROOT / "results/tables"

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

frame_metrics = pd.read_csv(FRAME_METRICS_PATH, sep="\t", low_memory=False)
threshold_df = pd.read_csv(THRESHOLD_PATH, sep="\t")
global_threshold_df = pd.read_csv(GLOBAL_THRESHOLD_PATH, sep="\t")
position_summary = pd.read_csv(POSITION_SUMMARY_PATH, sep="\t")
onset_summary = pd.read_csv(ONSET_SUMMARY_PATH, sep="\t")
parameters = json.loads(PARAMETERS_PATH.read_text())
background_trace_df = pd.read_csv(BACKGROUND_TRACE_PATH, sep="\t") if BACKGROUND_TRACE_PATH.exists() else pd.DataFrame()
flagged_background_df = pd.read_csv(BACKGROUND_OUTLIER_PATH, sep="\t") if BACKGROUND_OUTLIER_PATH.exists() else pd.DataFrame()
background_exclusion_window_df = (
    pd.read_csv(BACKGROUND_EXCLUSION_WINDOW_PATH, sep="\t")
    if BACKGROUND_EXCLUSION_WINDOW_PATH.exists()
    else pd.DataFrame()
)
flagged_position_qc = (
    flagged_background_df.sort_values("severity_score", ascending=False)
    .drop_duplicates("position_label")
    .reset_index(drop=True)
    if not flagged_background_df.empty
    else pd.DataFrame()
)
flagged_background_positions = (
    flagged_position_qc["position_label"].tolist()
    if not flagged_position_qc.empty
    else []
)
illumination_fields = None
if parameters.get("illumination_correction_applied") and parameters.get("illumination_field_path"):
    payload = np.load(Path(parameters["illumination_field_path"]))
    illumination_fields = {
        "RFP": payload["rfp_field"].astype(np.float32),
        "YFP": payload["yfp_field"].astype(np.float32),
    }

POSITION_RE = re.compile(r"Pos(?P<position_index>\d+)$")
REPORTER_COLORS = {"RFP": "tab:red", "YFP": "goldenrod"}
REPORTER_INDIVIDUAL_COLORS = {"RFP": "#d8a9a9", "YFP": "#cabb7a"}
REPORTER_LINESTYLES = {"RFP": "-", "YFP": "--"}
POPULATION_REPORTER_LINESTYLES = {"RFP": "-", "YFP": "-"}
POPULATION_THRESHOLD_SIGMAS = [float(value) for value in range(1, 9)]
POPULATION_BRIGHT_FRACTION = 0.10
POPULATION_DERIVATIVE_SMOOTH_WINDOW = 11
POPULATION_DERIVATIVE_SMOOTH_WINDOW_ALT = 21
POPULATION_DERIVATIVE_INTERVAL_HOURS = 2.0
POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT = 4.0
POPULATION_DERIVATIVE_WINDOW_SWEEP = [11, 21, 41]
POPULATION_DERIVATIVE_INTERVAL_SWEEP_HOURS = [2.0, 4.0, 8.0]
POPULATION_DERIVATIVE_GAUSSIAN_SIGMA_FRAMES = 8.0
TRACE_METRICS = [
    ("organoid_mean_intensity_z", "Whole-cyst mean intensity (sigma above pooled early within-cyst baseline)"),
    (
        "positive_fraction_sigma4",
        "Positive fraction within cyst mask (threshold = shared within-cyst baseline + 4 sigma)",
    ),
    (
        "brightest_decile_mean_intensity_z",
        "Brightest 10% of cyst pixels: mean intensity (sigma above pooled early within-cyst baseline)",
    ),
]

print("Project root:", ROOT)
print("Frame-metric rows:", len(frame_metrics))
print("Preview dir:", PREVIEW_DIR)
print("Interval minutes:", parameters["interval_minutes"])
print("Background estimator:", parameters.get("background_estimator", "annulus"))
print("Loaded background trace rows from 04b:", len(background_trace_df))
print("Loaded background outlier rows from 04b:", len(flagged_background_df))
print("Loaded background exclusion windows from 04b:", len(background_exclusion_window_df))
if "background_qc_exclude_from_analysis" in frame_metrics.columns:
    print(
        "Background-QC-excluded frame rows in 05 frame table:",
        int(frame_metrics["background_qc_exclude_from_analysis"].fillna(False).sum()),
    )
if flagged_background_df.empty and background_trace_df.empty:
    display(
        Markdown(
            "### Loaded Upstream Background QC\n\n"
            "Background QC outputs from notebook `04b_reporter_background_stability_qc.ipynb` were not found. "
            "Integrated review will still run, but without precomputed background flags."
        )
    )
else:
    upstream_lines = [
        "### Loaded Upstream Background QC",
        "",
        "Notebook `05` is using the precomputed background-QC outputs from notebook `04b`.",
        "",
        f"- flagged reporter-position traces: `{len(flagged_background_df)}`",
        f"- flagged positions (union across reporters): `{len(flagged_background_positions)}`",
        f"- explicit background cut windows: `{len(background_exclusion_window_df)}`",
        f"- cached whole-background trace rows: `{len(background_trace_df)}`",
    ]
    if "background_qc_exclude_from_analysis" in frame_metrics.columns:
        upstream_lines.extend(
            [
                "",
                "Background-QC windows are already applied in the frame metrics used below:",
                f"- frame rows excluded by background QC: `{int(frame_metrics['background_qc_exclude_from_analysis'].fillna(False).sum())}`",
                f"- affected positions: `{int(frame_metrics.loc[frame_metrics['background_qc_exclude_from_analysis'].fillna(False), 'position_label'].nunique())}`",
            ]
        )
    display(Markdown("\n".join(upstream_lines)))


### Define Helper Functions


In [ ]:
def position_index_from_label(position_label: str) -> int:
    match = POSITION_RE.fullmatch(position_label)
    if not match:
        raise ValueError(f"Unexpected position label: {position_label}")
    return int(match.group("position_index"))


def should_offset_time_column(column_name: str) -> bool:
    lower = column_name.lower()
    if "time" not in lower and "hour" not in lower:
        return False
    blocked_tokens = [
        "lag_",
        "minus_",
        "_minus",
        "delta",
        "diff",
        "rise_",
        "interval_",
        "duration",
    ]
    if any(token in lower for token in blocked_tokens):
        return False
    return True


def display_time_hours(values):
    return _display_time_hours(values, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours(value: float, decimals: int = 1) -> str:
    return _format_display_hours(value, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours_from_index(time_index: int | float, decimals: int = 1) -> str:
    return _format_display_hours_from_index(
        time_index,
        interval_hours=float(parameters["interval_minutes"]) / 60.0,
        decimals=decimals,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(ax, axis=axis, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    return _offset_time_hours_df(
        df,
        should_offset_column=should_offset_time_column,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def raw_frame_path(position_label: str, channel_index: int, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel{channel_index:03d}_position{position_index:03d}_time{time_index:09d}_z000.tif"
    )


def mask_path(position_label: str, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        MASK_ROOT
        / position_label
        / f"img_channel000_position{position_index:03d}_time{time_index:09d}_z000_mask.tiff"
    )


def load_phase(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(raw_frame_path(position_label, 0, time_index))


def load_reporter(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    channel_index = 1 if reporter == "RFP" else 2
    return tiff.imread(raw_frame_path(position_label, channel_index, time_index))


def load_reporter_after_illumination(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    image = load_reporter(position_label, reporter, time_index).astype(np.float32)
    if illumination_fields is None:
        return image
    return image / np.clip(illumination_fields[reporter], 1e-6, None)


def load_mask(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(mask_path(position_label, time_index)).astype(bool)


def display_image(image: np.ndarray, low_q: float = 1.0, high_q: float = 99.0) -> np.ndarray:
    low, high = np.percentile(image, [low_q, high_q])
    if math.isclose(high, low):
        high = low + 1.0
    return np.clip((image - low) / (high - low), 0, 1)


def draw_mask(ax, mask: np.ndarray, color: str = "deepskyblue", linewidth: float = 1.8) -> None:
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=linewidth)


def annulus_mask(
    organoid_mask: np.ndarray,
    inner: int,
    outer: int,
    min_ring_pixels: int,
) -> np.ndarray:
    inner_mask = morphology.binary_dilation(organoid_mask, morphology.disk(inner))
    outer_mask = morphology.binary_dilation(organoid_mask, morphology.disk(outer))
    ring = outer_mask & ~inner_mask
    if int(ring.sum()) < min_ring_pixels:
        ring = ~outer_mask
    if int(ring.sum()) < min_ring_pixels:
        ring = ~organoid_mask
    return ring


def background_reference_mask(
    organoid_mask: np.ndarray,
    background_estimator: str,
    background_ring_inner: int,
    background_ring_outer: int,
    min_ring_pixels: int,
) -> np.ndarray:
    if background_estimator == "whole_off_cyst":
        return ~organoid_mask
    return annulus_mask(
        organoid_mask=organoid_mask,
        inner=background_ring_inner,
        outer=background_ring_outer,
        min_ring_pixels=min_ring_pixels,
    )


def corrected_signal(
    signal: np.ndarray,
    organoid_mask: np.ndarray,
    background_estimator: str,
    background_ring_inner: int,
    background_ring_outer: int,
    min_ring_pixels: int,
) -> tuple[np.ndarray, float, np.ndarray]:
    background_mask = background_reference_mask(
        organoid_mask=organoid_mask,
        background_estimator=background_estimator,
        background_ring_inner=background_ring_inner,
        background_ring_outer=background_ring_outer,
        min_ring_pixels=min_ring_pixels,
    )
    if np.any(background_mask):
        background_value = float(np.median(signal[background_mask]))
    else:
        background_value = float(np.median(signal[~organoid_mask])) if np.any(~organoid_mask) else 0.0
    corrected = signal.astype(float) - background_value
    return corrected, background_value, background_mask


def gaussian_kernel1d_bins(sigma_bins: float) -> np.ndarray:
    sigma_bins = max(float(sigma_bins), 0.0)
    if sigma_bins <= 0:
        return np.array([1.0], dtype=float)
    radius = max(1, int(np.ceil(4.0 * sigma_bins)))
    x = np.arange(-radius, radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (x / sigma_bins) ** 2)
    kernel /= np.sum(kernel)
    return kernel


def smooth_histogram_counts(counts: np.ndarray, sigma_bins: float) -> np.ndarray:
    kernel = gaussian_kernel1d_bins(sigma_bins)
    if kernel.size == 1:
        return counts.astype(float)
    pad = kernel.size // 2
    padded = np.pad(counts.astype(float), (pad, pad), mode="edge")
    return np.convolve(padded, kernel, mode="valid")


def histogram_peak_diagnostics(
    values: np.ndarray,
    n_bins: int,
    smooth_sigma_bins: float,
) -> dict[str, object]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {
            "edges": np.array([0.0, 1.0], dtype=float),
            "centers": np.array([0.5], dtype=float),
            "counts": np.array([0.0], dtype=float),
            "smooth_counts": np.array([0.0], dtype=float),
            "raw_mode_center": float("nan"),
            "smoothed_peak": float("nan"),
            "bin_width": 1.0,
        }
    value_min = float(np.min(arr))
    value_max = float(np.max(arr))
    if not np.isfinite(value_min) or not np.isfinite(value_max):
        value_min, value_max = 0.0, 1.0
    if value_max <= value_min:
        value_max = value_min + 1.0
    bins = max(32, int(n_bins))
    edges = np.linspace(value_min, value_max, bins + 1, dtype=float)
    counts, _ = np.histogram(arr, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    smooth_counts = smooth_histogram_counts(counts, smooth_sigma_bins)
    raw_mode_idx = int(np.nanargmax(counts))
    smoothed_mode_idx = int(np.nanargmax(smooth_counts))
    return {
        "edges": edges,
        "centers": centers,
        "counts": counts.astype(float),
        "smooth_counts": smooth_counts.astype(float),
        "raw_mode_center": float(centers[raw_mode_idx]),
        "smoothed_peak": float(centers[smoothed_mode_idx]),
        "bin_width": float(edges[1] - edges[0]),
    }


def half_gaussian_fit_counts(
    centers: np.ndarray,
    counts: np.ndarray,
    peak_location: float,
    sigma: float,
) -> np.ndarray:
    x = np.asarray(centers, dtype=float)
    y = np.asarray(counts, dtype=float)
    sigma = max(float(sigma), 1e-6)
    basis = np.exp(-0.5 * ((x - float(peak_location)) / sigma) ** 2)
    left_mask = x <= float(peak_location)
    if not np.any(left_mask):
        amplitude = float(np.nanmax(y)) if y.size else 0.0
    else:
        left_basis = basis[left_mask]
        left_counts = y[left_mask]
        denom = float(np.dot(left_basis, left_basis))
        if denom <= 0:
            amplitude = float(np.nanmax(left_counts)) if left_counts.size else 0.0
        else:
            amplitude = max(0.0, float(np.dot(left_counts, left_basis) / denom))
    return amplitude * basis


def annulus_statistics(
    signal: np.ndarray,
    organoid_mask: np.ndarray,
    background_ring_inner: int,
    background_ring_outer: int,
    min_ring_pixels: int,
) -> dict[str, object]:
    ring = annulus_mask(
        organoid_mask=organoid_mask,
        inner=background_ring_inner,
        outer=background_ring_outer,
        min_ring_pixels=min_ring_pixels,
    )
    if np.any(ring):
        values = signal[ring].astype(float)
    else:
        values = signal[~organoid_mask].astype(float) if np.any(~organoid_mask) else np.array([], dtype=float)
    if values.size:
        background_value = float(np.median(values))
        q25, q75 = np.quantile(values, [0.25, 0.75])
        mad = float(np.median(np.abs(values - background_value)))
    else:
        background_value = float("nan")
        q25 = float("nan")
        q75 = float("nan")
        mad = float("nan")
    return {
        "ring": ring,
        "values": values,
        "background_value": background_value,
        "iqr": float(q75 - q25) if np.isfinite(q25) and np.isfinite(q75) else float("nan"),
        "mad": mad,
    }


def whole_background_statistics(
    signal: np.ndarray,
    organoid_mask: np.ndarray,
) -> dict[str, object]:
    values = signal[~organoid_mask].astype(float) if np.any(~organoid_mask) else np.array([], dtype=float)
    if values.size:
        background_value = float(np.median(values))
        q25, q75 = np.quantile(values, [0.25, 0.75])
        mad = float(np.median(np.abs(values - background_value)))
    else:
        background_value = float("nan")
        q25 = float("nan")
        q75 = float("nan")
        mad = float("nan")
    return {
        "values": values,
        "background_value": background_value,
        "iqr": float(q75 - q25) if np.isfinite(q25) and np.isfinite(q75) else float("nan"),
        "mad": mad,
    }


def positive_mask_from_corrected(
    corrected: np.ndarray,
    organoid_mask: np.ndarray,
    threshold_value: float,
) -> np.ndarray:
    positive = organoid_mask & (corrected > float(threshold_value))
    closing_radius = int(parameters.get("positive_closing_radius", 0))
    min_object_size = int(parameters.get("positive_min_object_size", 1))
    hole_area = int(parameters.get("positive_hole_area", 1))
    if closing_radius > 0:
        positive = morphology.binary_closing(positive, morphology.disk(closing_radius))
    if min_object_size > 1:
        positive = morphology.remove_small_objects(positive, min_size=min_object_size)
    if hole_area > 1:
        positive = morphology.remove_small_holes(positive, area_threshold=hole_area)
    return positive & organoid_mask


def brightest_fraction_mean(values: np.ndarray, fraction: float) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    keep_n = max(1, int(np.ceil(float(fraction) * arr.size)))
    partitioned = np.partition(arr, arr.size - keep_n)
    return float(np.mean(partitioned[-keep_n:]))


def rolling_median_smooth(values: np.ndarray, window: int) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr.copy()
    return (
        pd.Series(arr)
        .rolling(window=max(1, int(window)), center=True, min_periods=1)
        .median()
        .to_numpy(dtype=float)
    )


def rolling_mean_smooth(values: np.ndarray, window: int) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr.copy()
    return (
        pd.Series(arr)
        .rolling(window=max(1, int(window)), center=True, min_periods=1)
        .mean()
        .to_numpy(dtype=float)
    )


def gaussian_smooth(values: np.ndarray, sigma_frames: float) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr.copy()
    sigma = float(sigma_frames)
    if not np.isfinite(sigma) or sigma <= 0:
        return arr.copy()
    radius = max(1, int(np.ceil(3.0 * sigma)))
    x = np.arange(-radius, radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel /= np.sum(kernel)

    finite_mask = np.isfinite(arr)
    if finite_mask.sum() == 0:
        return np.full_like(arr, np.nan, dtype=float)
    valid_index = np.flatnonzero(finite_mask)
    valid_values = arr[finite_mask]
    filled = np.interp(np.arange(arr.size), valid_index, valid_values)
    padded = np.pad(filled, pad_width=radius, mode="edge")
    smoothed = np.convolve(padded, kernel, mode="valid")
    smoothed[~finite_mask] = np.nan
    return smoothed


def derivative_over_time(values: np.ndarray, time_hours: np.ndarray) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    time_arr = np.asarray(time_hours, dtype=float)
    if arr.size < 2 or time_arr.size < 2:
        return np.full_like(arr, np.nan, dtype=float)
    if not np.isfinite(arr).any() or not np.isfinite(time_arr).all():
        return np.full_like(arr, np.nan, dtype=float)
    return np.gradient(arr, time_arr)


def centered_slope_over_interval(
    values: np.ndarray,
    time_hours: np.ndarray,
    interval_hours: float,
) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    time_arr = np.asarray(time_hours, dtype=float)
    if arr.size < 3 or time_arr.size < 3:
        return np.full_like(arr, np.nan, dtype=float)
    if not np.isfinite(arr).any() or not np.isfinite(time_arr).all():
        return np.full_like(arr, np.nan, dtype=float)

    interval = float(interval_hours)
    if not np.isfinite(interval) or interval <= 0:
        return np.full_like(arr, np.nan, dtype=float)

    slope = np.full_like(arr, np.nan, dtype=float)
    finite_mask = np.isfinite(arr)
    if finite_mask.sum() < 3:
        return slope

    finite_times = time_arr[finite_mask]
    finite_values = arr[finite_mask]
    t_min = float(np.min(finite_times))
    t_max = float(np.max(finite_times))
    half_interval = 0.5 * interval
    for idx, t in enumerate(time_arr):
        if not np.isfinite(arr[idx]):
            continue
        left_t = t - half_interval
        right_t = t + half_interval
        if left_t < t_min or right_t > t_max:
            continue
        left_v = float(np.interp(left_t, finite_times, finite_values))
        right_v = float(np.interp(right_t, finite_times, finite_values))
        slope[idx] = (right_v - left_v) / interval
    return slope


def summarize_population_by_time(
    population_df: pd.DataFrame,
    value_col: str,
) -> pd.DataFrame:
    rows = []
    for (reporter, time_hours), subset in population_df.groupby(["reporter", "time_hours"], sort=True):
        values = subset[value_col].astype(float).to_numpy()
        finite = values[np.isfinite(values)]
        if finite.size == 0:
            rows.append(
                {
                    "reporter": reporter,
                    "time_hours": float(time_hours),
                    "mean": float("nan"),
                    "median": float("nan"),
                    "std": float("nan"),
                    "sem": float("nan"),
                    "q25": float("nan"),
                    "q75": float("nan"),
                    "count": 0,
                }
            )
            continue
        mean_value = float(np.mean(finite))
        std_value = float(np.std(finite, ddof=1)) if finite.size > 1 else 0.0
        rows.append(
            {
                "reporter": reporter,
                "time_hours": float(time_hours),
                "mean": mean_value,
                "median": float(np.median(finite)),
                "std": std_value,
                "sem": float(std_value / np.sqrt(finite.size)) if finite.size > 0 else float("nan"),
                "q25": float(np.quantile(finite, 0.25)),
                "q75": float(np.quantile(finite, 0.75)),
                "count": int(finite.size),
            }
        )
    return pd.DataFrame(rows).sort_values(["reporter", "time_hours"]).reset_index(drop=True)


def focus_ylim_from_summary(
    summary_df: pd.DataFrame,
    include_zero: bool = False,
    padding_fraction: float = 0.18,
) -> tuple[float, float]:
    available_cols = [col for col in ["mean", "q25", "q75", "median"] if col in summary_df.columns]
    subset = summary_df.loc[:, available_cols].astype(float)
    values = subset.to_numpy(dtype=float).ravel()
    values = values[np.isfinite(values)]
    if values.size == 0:
        return (0.0, 1.0)
    center_col = "mean" if "mean" in subset.columns else "median"
    lower_partner = "q25" if "q25" in subset.columns else center_col
    upper_partner = "q75" if "q75" in subset.columns else center_col
    lower = float(np.nanmin(np.concatenate([
        subset[center_col].to_numpy(dtype=float)[np.isfinite(subset[center_col].to_numpy(dtype=float))],
        subset[lower_partner].to_numpy(dtype=float)[np.isfinite(subset[lower_partner].to_numpy(dtype=float))],
    ])))
    upper = float(np.nanmax(np.concatenate([
        subset[center_col].to_numpy(dtype=float)[np.isfinite(subset[center_col].to_numpy(dtype=float))],
        subset[upper_partner].to_numpy(dtype=float)[np.isfinite(subset[upper_partner].to_numpy(dtype=float))],
    ])))
    if include_zero:
        lower = min(lower, 0.0)
        upper = max(upper, 0.0)
    span = upper - lower
    if not np.isfinite(span) or span <= 0:
        span = max(abs(upper), 1.0)
    pad = max(1e-6, float(padding_fraction) * span)
    return (lower - pad, upper + pad)


def symlog_linthresh(values: np.ndarray, min_linthresh: float = 0.05) -> float:
    arr = np.asarray(values, dtype=float)
    arr = np.abs(arr[np.isfinite(arr)])
    if arr.size == 0:
        return float(min_linthresh)
    positive = arr[arr > 0]
    if positive.size == 0:
        return float(min_linthresh)
    return float(max(min_linthresh, np.quantile(positive, 0.10)))


def example_corrected_display_limits(
    bundles: list[dict[str, object]],
    low_q: float = 0.01,
    high_q: float = 0.995,
) -> tuple[float, float]:
    sampled_values = []
    for bundle in bundles:
        sampled_values.append(np.asarray(bundle["early_corrected"], dtype=np.float32).ravel())
    pooled = np.concatenate(sampled_values) if sampled_values else np.array([0.0, 1.0], dtype=np.float32)
    vmin = float(np.quantile(pooled, low_q))
    vmax = float(np.quantile(pooled, high_q))
    if math.isclose(vmax, vmin):
        vmax = vmin + 1.0
    return vmin, vmax


def corrected_display_limits_from_arrays(
    arrays: list[np.ndarray],
    low_q: float = 0.01,
    high_q: float = 0.995,
) -> tuple[float, float]:
    pooled = np.concatenate(
        [np.asarray(array, dtype=np.float32).ravel() for array in arrays]
    ) if arrays else np.array([0.0, 1.0], dtype=np.float32)
    vmin = float(np.quantile(pooled, low_q))
    vmax = float(np.quantile(pooled, high_q))
    if math.isclose(vmax, vmin):
        vmax = vmin + 1.0
    return vmin, vmax


def position_display_limits(position_label: str, sample_n: int = 12) -> tuple[float, float]:
    subset = frame_metrics.loc[
        (frame_metrics["position_label"] == position_label)
        & (frame_metrics["reporter"] == "RFP")
    ].sort_values("time_index")
    frames = subset["time_index"].astype(int).tolist()
    if not frames:
        return 0.0, 1.0
    if len(frames) <= sample_n:
        sample_frames = frames
    else:
        indices = np.linspace(0, len(frames) - 1, sample_n).round().astype(int)
        sample_frames = [frames[idx] for idx in indices]

    lows = []
    highs = []
    for time_index in sample_frames:
        image = load_phase(position_label, int(time_index))
        low, high = np.percentile(image, [1.0, 99.0])
        lows.append(low)
        highs.append(high)

    low = float(np.median(lows))
    high = float(np.median(highs))
    if math.isclose(high, low):
        high = low + 1.0
    return low, high


def baseline_time_indices(position_label: str, reporter: str) -> list[int]:
    baseline_count = int(
        threshold_df.loc[
            (threshold_df["position_label"] == position_label)
            & (threshold_df["reporter"] == reporter),
            "baseline_frame_count",
        ].iloc[0]
    )
    subset = frame_metrics.loc[
        (frame_metrics["position_label"] == position_label)
        & (frame_metrics["reporter"] == reporter)
        & (~frame_metrics["exclude_from_analysis"])
    ].sort_values("time_index")
    return subset["time_index"].astype(int).head(baseline_count).tolist()


def threshold_example_bundle(position_label: str, reporter: str) -> dict[str, object]:
    threshold_row = threshold_df.loc[
        (threshold_df["position_label"] == position_label)
        & (threshold_df["reporter"] == reporter)
    ].iloc[0]
    global_threshold_row = global_threshold_df.loc[global_threshold_df["reporter"] == reporter].iloc[0]
    baseline_times = baseline_time_indices(position_label, reporter)
    if not baseline_times:
        raise RuntimeError(f"No baseline times available for {position_label}")

    pooled_values = []
    for time_index in baseline_times:
        mask = load_mask(position_label, time_index)
        reporter_image = load_reporter_after_illumination(position_label, reporter, time_index).astype(float)
        corrected, _, _ = corrected_signal(
            signal=reporter_image,
            organoid_mask=mask,
            background_estimator=str(parameters.get("background_estimator", "whole_off_cyst")),
            background_ring_inner=int(parameters["background_ring_inner"]),
            background_ring_outer=int(parameters["background_ring_outer"]),
            min_ring_pixels=int(parameters["min_ring_pixels"]),
        )
        pooled_values.append(corrected[mask])

    pooled = np.concatenate(pooled_values) if pooled_values else np.array([], dtype=float)
    cutoff = float(global_threshold_row.get("null_partition_value", global_threshold_row["global_cutoff"]))
    baseline_pooled = pooled[pooled <= cutoff] if pooled.size else np.array([], dtype=float)
    baseline_location = float(threshold_row["baseline_location"])
    baseline_scale = float(threshold_row["baseline_scale"])
    threshold_value = float(threshold_row["threshold_value"])
    pooled_score = (pooled - baseline_location) / baseline_scale if pooled.size else np.array([], dtype=float)
    cutoff_score = (cutoff - baseline_location) / baseline_scale if pooled.size else float("nan")
    threshold_score = (threshold_value - baseline_location) / baseline_scale

    early_time = int(baseline_times[min(len(baseline_times) // 2, len(baseline_times) - 1)])
    early_mask = load_mask(position_label, early_time)
    early_signal = load_reporter_after_illumination(position_label, reporter, early_time).astype(float)
    early_corrected, early_background, early_background_mask = corrected_signal(
        signal=early_signal,
        organoid_mask=early_mask,
        background_estimator=str(parameters.get("background_estimator", "whole_off_cyst")),
        background_ring_inner=int(parameters["background_ring_inner"]),
        background_ring_outer=int(parameters["background_ring_outer"]),
        min_ring_pixels=int(parameters["min_ring_pixels"]),
    )
    early_score = (early_corrected - baseline_location) / baseline_scale
    early_baseline_mask = early_mask & (early_corrected <= cutoff)

    trace_subset = frame_metrics.loc[
        (frame_metrics["position_label"] == position_label)
        & (frame_metrics["reporter"] == reporter)
        & (~frame_metrics["exclude_from_analysis"])
    ].sort_values("time_index").copy()
    trace_subset["threshold_value"] = float(threshold_value)

    available_times = trace_subset["time_index"].astype(int).tolist()

    def nearest_available_time(target_time: float, used: set[int]) -> int:
        ordered = sorted(available_times, key=lambda t: (abs(t - target_time), t))
        for time_index in ordered:
            if time_index not in used:
                return int(time_index)
        return int(ordered[0])

    review_times: list[int] = []
    used_times: set[int] = set()
    early_review_time = nearest_available_time(9, used_times)
    review_times.append(early_review_time)
    used_times.add(early_review_time)
    final_time = int(available_times[-1])
    later_targets = [
        early_review_time + (final_time - early_review_time) * frac
        for frac in (1 / 3, 2 / 3, 1.0)
    ]
    for target_time in later_targets:
        selected_time = nearest_available_time(target_time, used_times)
        review_times.append(selected_time)
        used_times.add(selected_time)

    review_frames = []
    for time_index in review_times:
        review_mask = load_mask(position_label, time_index)
        review_signal = load_reporter_after_illumination(position_label, reporter, time_index).astype(float)
        review_corrected, review_background, _ = corrected_signal(
            signal=review_signal,
            organoid_mask=review_mask,
            background_estimator=str(parameters.get("background_estimator", "whole_off_cyst")),
            background_ring_inner=int(parameters["background_ring_inner"]),
            background_ring_outer=int(parameters["background_ring_outer"]),
            min_ring_pixels=int(parameters["min_ring_pixels"]),
        )
        review_raw_positive_mask = review_mask & (review_corrected > threshold_value)
        review_positive_mask = positive_mask_from_corrected(
            corrected=review_corrected,
            organoid_mask=review_mask,
            threshold_value=threshold_value,
        )
        positive_fraction = (
            float(np.sum(review_positive_mask) / np.sum(review_mask))
            if np.sum(review_mask)
            else float("nan")
        )
        review_frames.append(
            {
                "time_index": int(time_index),
                "corrected": review_corrected,
                "mask": review_mask,
                "background_value": float(review_background),
                "raw_positive_mask": review_raw_positive_mask,
                "positive_mask": review_positive_mask,
                "positive_fraction": positive_fraction,
            }
        )

    return {
        "position_label": position_label,
        "reporter": reporter,
        "threshold_value": threshold_value,
        "baseline_location": baseline_location,
        "baseline_scale": baseline_scale,
        "cutoff": cutoff,
        "pooled_q95": float(np.quantile(pooled, 0.95)) if pooled.size else float("nan"),
        "fraction_below_cutoff": float(baseline_pooled.size / pooled.size) if pooled.size else float("nan"),
        "pooled_score": pooled_score,
        "cutoff_score": float(cutoff_score),
        "threshold_score": float(threshold_score),
        "pooled": pooled,
        "baseline_pooled": baseline_pooled,
        "early_time": early_time,
        "early_mask": early_mask,
        "early_background_mask": early_background_mask,
        "early_corrected": early_corrected,
        "early_score": early_score,
        "early_baseline_mask": early_baseline_mask,
        "early_background": float(early_background),
        "review_frames": review_frames,
        "trace_subset": trace_subset,
    }


def global_threshold_diagnostic_bundle(reporter: str) -> dict[str, object]:
    global_row = global_threshold_df.loc[global_threshold_df["reporter"] == reporter].iloc[0]
    pooled_values = []
    for position_label in threshold_df.loc[threshold_df["reporter"] == reporter, "position_label"].tolist():
        baseline_times = baseline_time_indices(position_label, reporter)
        for time_index in baseline_times:
            mask = load_mask(position_label, time_index)
            reporter_image = load_reporter_after_illumination(position_label, reporter, time_index).astype(float)
            corrected, _, _ = corrected_signal(
                signal=reporter_image,
                organoid_mask=mask,
                background_estimator=str(parameters.get("background_estimator", "whole_off_cyst")),
                background_ring_inner=int(parameters["background_ring_inner"]),
                background_ring_outer=int(parameters["background_ring_outer"]),
                min_ring_pixels=int(parameters["min_ring_pixels"]),
            )
            pooled_values.append(corrected[mask])

    pooled = np.concatenate(pooled_values) if pooled_values else np.array([], dtype=float)
    cutoff = float(global_row.get("null_partition_value", global_row["global_cutoff"]))
    baseline_pooled = pooled[pooled <= cutoff] if pooled.size else np.array([], dtype=float)
    peak_hist_bins = int(global_row["baseline_peak_hist_bins"])
    peak_smooth_sigma_bins = float(global_row["baseline_peak_smooth_sigma_bins"])
    peak_diagnostics = histogram_peak_diagnostics(
        pooled,
        n_bins=peak_hist_bins,
        smooth_sigma_bins=peak_smooth_sigma_bins,
    )
    peak_diagnostics["half_gaussian_counts"] = half_gaussian_fit_counts(
        peak_diagnostics["centers"],
        peak_diagnostics["counts"],
        float(global_row["baseline_location"]),
        float(global_row["baseline_scale"]),
    )
    return {
        "reporter": reporter,
        "pooled": pooled,
        "baseline_pooled": baseline_pooled,
        "cutoff": cutoff,
        "baseline_location": float(global_row["baseline_location"]),
        "baseline_scale": float(global_row["baseline_scale"]),
        "threshold_value": float(global_row["threshold_value"]),
        "baseline_frame_count_per_position": int(global_row["baseline_frame_count_per_position"]),
        "pooled_position_count": int(global_row["pooled_position_count"]),
        "pooled_frame_count": int(global_row["pooled_frame_count"]),
        "pooled_pixel_count": int(global_row["pooled_pixel_count"]),
        "peak_hist_bins": peak_hist_bins,
        "peak_smooth_sigma_bins": peak_smooth_sigma_bins,
        "peak_diagnostics": peak_diagnostics,
        "threshold_scope": str(global_row["threshold_scope"]),
    }


def chunked(sequence: list[tuple[str, str]], chunk_size: int) -> list[list[tuple[str, str]]]:
    chunk_size = max(1, int(chunk_size))
    return [sequence[idx : idx + chunk_size] for idx in range(0, len(sequence), chunk_size)]


In [ ]:
def select_background_problem_times(summary_row: pd.Series, time_indices: list[int]) -> list[int]:
    if not time_indices:
        return []
    issue_class = str(summary_row["issue_class"])
    first_flag_time = summary_row.get("position_first_flag_time_index", np.nan)
    last_flag_time = summary_row.get("position_last_flag_time_index", np.nan)
    if not np.isfinite(first_flag_time):
        first_flag_time = summary_row.get("first_flag_time_index", np.nan)
    if not np.isfinite(last_flag_time):
        last_flag_time = summary_row.get("last_flag_time_index", np.nan)
    if np.isfinite(first_flag_time) and int(first_flag_time) in time_indices:
        start_idx = time_indices.index(int(first_flag_time))
        if np.isfinite(last_flag_time) and int(last_flag_time) in time_indices:
            end_idx = time_indices.index(int(last_flag_time))
        else:
            end_idx = start_idx
        midpoint_idx = int(round((start_idx + end_idx) / 2))
        candidate_indices = [
            max(0, start_idx - 1),
            start_idx,
            midpoint_idx,
            min(len(time_indices) - 1, end_idx + 1),
        ]
    elif issue_class == "sudden jump":
        prev_time = int(summary_row["max_step_time_prev"])
        curr_time = int(summary_row["max_step_time_curr"])
        if prev_time in time_indices:
            prev_idx = time_indices.index(prev_time)
        else:
            prev_idx = max(0, len(time_indices) // 2 - 1)
        candidate_indices = [max(0, prev_idx - 1), prev_idx, min(len(time_indices) - 1, prev_idx + 1), min(len(time_indices) - 1, prev_idx + 2)]
    else:
        candidate_indices = np.linspace(0, len(time_indices) - 1, 4).round().astype(int).tolist()
    selected = []
    for idx in candidate_indices:
        time_index = int(time_indices[idx])
        if time_index not in selected:
            selected.append(time_index)
    return selected


def whole_background_trace_df(
    position_label: str,
    reporter: str,
) -> pd.DataFrame:
    if "background_trace_df" in globals() and not background_trace_df.empty:
        cached = background_trace_df.loc[
            (background_trace_df["position_label"] == position_label)
            & (background_trace_df["reporter"] == reporter)
        ].sort_values("time_index")
        if not cached.empty:
            return cached.reset_index(drop=True)
    subset = frame_metrics.loc[
        (frame_metrics["position_label"] == position_label)
        & (frame_metrics["reporter"] == reporter)
        & (~frame_metrics["exclude_from_analysis"])
    ].sort_values("time_index")
    rows = []
    for _, row in subset.iterrows():
        time_index = int(row["time_index"])
        mask = load_mask(position_label, time_index)
        signal = load_reporter_after_illumination(position_label, reporter, time_index).astype(float)
        bg_info = whole_background_statistics(
            signal=signal,
            organoid_mask=mask,
        )
        rows.append(
            {
                "position_label": position_label,
                "reporter": reporter,
                "time_index": time_index,
                "time_hours": float(row["time_hours"]),
                "background_value": float(bg_info["background_value"]),
            }
        )
    return pd.DataFrame(rows)


## Review Reporter Thresholds

This section should answer four concrete questions directly in the notebook:

1. how the local image background is measured
2. how the RFP-negative / YFP-negative tissue baseline is defined
3. how thresholded reporter pixels are turned into contiguous positive regions
4. whether the same threshold is reused across the timelapse while the local background changes frame to frame

Important distinction:

- the whole off-cyst background is the per-frame background subtraction value
- the threshold plots below show the separate low-threshold positivity cutoffs used after that subtraction
- the low-threshold positivity cutoff is now shared within each reporter across positions
- the per-position image panels are still useful, because they show how that shared reporter threshold behaves on real cysts over time


In [ ]:
threshold_z = float(global_threshold_df["threshold_z"].iloc[0])
baseline_frame_count = int(global_threshold_df["baseline_frame_count_per_position"].iloc[0])
threshold_scope = str(global_threshold_df["threshold_scope"].iloc[0])


In [ ]:
display(
    Markdown(
        f'''
        ### Threshold definition used in this run

        For each **reporter** separately:

        1. For every position, take the first **{baseline_frame_count} clean frames**.
        2. Apply the masked illumination correction from notebook 04.
        3. Measure background from the **whole off-cyst region** and subtract it.
        4. Pool all corrected reporter pixels **inside the organoid mask** across positions and early frames.
        5. Estimate the shared within-cyst baseline center as the **smoothed mode of the full pooled early within-cyst histogram**.
        6. Estimate the shared within-cyst sigma from the **left half of that pooled histogram**.
        7. Set the threshold as:

        `shared reporter threshold = pooled early within-cyst baseline peak + ({threshold_z:.1f} x sigma)`

        Important distinction:

        - the **whole off-cyst background** is remeasured on **every frame**
        - the **reporter threshold** is fixed for that **reporter** and then reused across all positions and timepoints
        - the global-threshold scope for this run is: `{threshold_scope}`
        - the first image panels below are shown in **corrected intensity** units, not score units
        - those corrected-intensity image panels still use one **fixed display range per chunk**, estimated once from the **early corrected images shown in that chunk**, so positions with similar early brightness stay visible together
        - the green early-frame overlay shows the **exact pixels** that fall at or below the shared within-cyst baseline peak and therefore lie on the left-half side used for sigma estimation
        - the baseline **peak** is estimated from the smoothed full pooled histogram using **edge padding**
        - the baseline **sigma** is estimated from the **left half of the full pooled histogram** using the weighted second moment, following the pSMAD half-Gaussian logic

        The figures below first show the pooled global reporter histograms, then concrete per-position examples using that shared threshold.
        '''
    )
)

threshold_examples = {}
global_bundles = {reporter: global_threshold_diagnostic_bundle(reporter) for reporter in REPORTER_COLORS}
fig, axes = plt.subplots(2, 3, figsize=(24, 9), constrained_layout=True, squeeze=False)
for row_index, (reporter, color) in enumerate(REPORTER_COLORS.items()):
    bundle = global_bundles[reporter]
    peak_diag = bundle["peak_diagnostics"]
    raw_mode_center = float(peak_diag["raw_mode_center"])
    left_quantile = float(np.quantile(bundle["baseline_pooled"], 0.01)) if bundle["baseline_pooled"].size else raw_mode_center - 4.0 * bundle["baseline_scale"]
    right_quantile = float(np.quantile(bundle["baseline_pooled"], 0.999)) if bundle["baseline_pooled"].size else raw_mode_center + 4.0 * bundle["baseline_scale"]
    context_half_width = max(
        raw_mode_center - left_quantile,
        right_quantile - raw_mode_center,
        abs(float(bundle["threshold_value"]) - raw_mode_center),
        5.6 * float(bundle["baseline_scale"]),
    )
    context_half_width *= 1.10
    context_xmin = raw_mode_center - context_half_width
    context_xmax = raw_mode_center + context_half_width
    if math.isclose(context_xmax, context_xmin):
        context_xmax = context_xmin + 1.0
    retained_half_width = max(
        raw_mode_center - left_quantile,
        right_quantile - raw_mode_center,
        abs(float(bundle["baseline_location"]) - raw_mode_center),
        3.5 * float(bundle["baseline_scale"]),
    )
    retained_half_width *= 1.05
    retained_xmin = raw_mode_center - retained_half_width
    retained_xmax = raw_mode_center + retained_half_width
    if math.isclose(retained_xmax, retained_xmin):
        retained_xmax = retained_xmin + 1.0
    ax = axes[row_index, 0]
    ax.hist(bundle["pooled"], bins=120, color="0.75", edgecolor="white")
    ax.axvline(bundle["baseline_location"], color="deepskyblue", linestyle="--", linewidth=2.0, label="Shared within-cyst baseline peak / left-half partition")
    ax.axvline(bundle["threshold_value"], color="magenta", linewidth=2.4, label="Shared reporter threshold")
    ax.set_title(f"{reporter} pooled early corrected-intensity distribution (broader null-region view)")
    ax.set_xlabel("Corrected intensity")
    ax.set_ylabel("Pixel count")
    ax.set_xlim(context_xmin, context_xmax)
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.03,
        0.97,
        f"positions={bundle['pooled_position_count']} | frames/position={bundle['baseline_frame_count_per_position']} | pooled pixels={bundle['pooled_pixel_count']}\npeak={bundle['baseline_location']:.0f} | sigma={bundle['baseline_scale']:.0f} | low thr={bundle['threshold_value']:.0f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=2.0),
    )

    ax = axes[row_index, 1]
    ax.bar(
        peak_diag["centers"],
        peak_diag["counts"],
        width=peak_diag["bin_width"],
        color="0.60",
        edgecolor="white",
        linewidth=0.2,
        align="center",
    )
    ax.axvline(peak_diag["raw_mode_center"], color="tab:orange", linestyle="--", linewidth=2.0, label="Highest raw-count bin")
    ax.axvline(bundle["baseline_location"], color="deepskyblue", linestyle="--", linewidth=2.0, label="Shared within-cyst baseline peak / left-half partition")
    ax.set_title(f"{reporter} pooled pixels at or below the shared within-cyst baseline peak")
    ax.set_xlabel("Corrected intensity")
    ax.set_ylabel("Pixel count")
    ax.set_xlim(retained_xmin, retained_xmax)
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.03,
        0.97,
        f"left-half pixels={bundle['baseline_pooled'].size} | exact full-hist bins={bundle['peak_hist_bins']} | bin width={peak_diag['bin_width']:.1f}\nraw-mode bin={peak_diag['raw_mode_center']:.0f} | smoothed peak={bundle['baseline_location']:.0f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=2.0),
    )

    ax = axes[row_index, 2]
    ax.bar(
        peak_diag["centers"],
        peak_diag["counts"],
        width=peak_diag["bin_width"],
        color="0.78",
        edgecolor="white",
        linewidth=0.2,
        align="center",
        label="Raw full-hist bin counts",
    )
    ax.plot(
        peak_diag["centers"],
        peak_diag["smooth_counts"],
        color="black",
        linewidth=2.0,
        label=f"Gaussian-smoothed counts (sigma={bundle['peak_smooth_sigma_bins']:.1f} bins)",
    )
    ax.plot(
        peak_diag["centers"],
        peak_diag["half_gaussian_counts"],
        color="royalblue",
        linewidth=2.0,
        linestyle="--",
        label="Half-Gaussian null fit (left side reflected)",
    )
    ax.axvline(peak_diag["raw_mode_center"], color="tab:orange", linestyle="--", linewidth=2.0, label="Highest raw-count bin")
    ax.axvline(bundle["baseline_location"], color="deepskyblue", linestyle="--", linewidth=2.0, label="Shared within-cyst baseline peak")
    ax.axvline(bundle["baseline_location"] - bundle["baseline_scale"], color="gold", linestyle=":", linewidth=1.6, label="Peak +/- sigma")
    ax.axvline(bundle["baseline_location"] + bundle["baseline_scale"], color="gold", linestyle=":", linewidth=1.6)
    ax.set_title(f"{reporter} peak-estimation diagnostics from full pooled histogram")
    ax.set_xlabel("Corrected intensity (same 256-bin centers)")
    ax.set_ylabel("Histogram count")
    ax.set_xlim(retained_xmin, retained_xmax)
    ax.legend(loc="upper right", fontsize=8)

threshold_fig_path = FIGURE_DIR / "05_reporter_threshold_summary.png"
fig.savefig(threshold_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote threshold summary figure:", threshold_fig_path)

threshold_example_reporters = ["RFP", "YFP"]
for reporter in threshold_example_reporters:
    rep_positions = threshold_df.loc[threshold_df["reporter"] == reporter, "position_label"].tolist()
    full_example_bundles = [threshold_example_bundle(position_label, reporter) for position_label in rep_positions]
    ordered_pairs = sorted(
        zip(rep_positions, full_example_bundles),
        key=lambda item: (item[1]["pooled_q95"], item[0]),
    )
    ordered_positions = [position_label for position_label, _ in ordered_pairs]
    bundle_lookup = {position_label: bundle for position_label, bundle in ordered_pairs}
    if reporter == "RFP" and DEBUG_SHOW_ALL_RFP_THRESHOLD_EXAMPLES:
        full_example_positions = [(f"brightness rank {idx + 1}", position_label) for idx, position_label in enumerate(ordered_positions)]
        example_position_chunks = chunked(full_example_positions, DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE)
    elif reporter == "YFP" and DEBUG_SHOW_ALL_YFP_THRESHOLD_EXAMPLES:
        full_example_positions = [(f"brightness rank {idx + 1}", position_label) for idx, position_label in enumerate(ordered_positions)]
        example_position_chunks = chunked(full_example_positions, DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE)
    elif reporter == "RFP":
        index_specs = [
            ("low early brightness", 0),
            ("lower-mid early brightness", max(1, len(ordered_positions) // 4)),
            ("typical early brightness", len(ordered_positions) // 2),
            ("upper-mid early brightness", min(len(ordered_positions) - 2, (3 * len(ordered_positions)) // 4)),
            ("high early brightness", len(ordered_positions) - 1),
        ]
        seen = set()
        full_example_positions = []
        for label, idx in index_specs:
            idx = max(0, min(len(ordered_positions) - 1, int(idx)))
            position_label = ordered_positions[idx]
            if position_label in seen:
                continue
            seen.add(position_label)
            full_example_positions.append((label, position_label))
        example_position_chunks = [full_example_positions]
    else:
        index_specs = [
            ("low early brightness", 0),
            ("lower-mid early brightness", max(1, len(ordered_positions) // 4)),
            ("typical early brightness", len(ordered_positions) // 2),
            ("upper-mid early brightness", min(len(ordered_positions) - 2, (3 * len(ordered_positions)) // 4)),
            ("high early brightness", len(ordered_positions) - 1),
        ]
        seen = set()
        full_example_positions = []
        for label, idx in index_specs:
            idx = max(0, min(len(ordered_positions) - 1, int(idx)))
            position_label = ordered_positions[idx]
            if position_label in seen:
                continue
            seen.add(position_label)
            full_example_positions.append((label, position_label))
        example_position_chunks = [full_example_positions]

    full_example_bundles = [bundle_lookup[position_label] for _, position_label in full_example_positions]
    typical_index = next((
        idx for idx, (label, _) in enumerate(full_example_positions) if label == "typical early brightness"
    ), len(full_example_positions) // 2)
    threshold_examples[reporter] = {
        "low": full_example_positions[0][1],
        "typical": full_example_positions[typical_index][1],
        "high": full_example_positions[-1][1],
        "ordered_positions": ordered_positions,
    }
    threshold_example_selection_df = pd.DataFrame(
        [
            {
                "selection_rank": int(selection_rank),
                "selection_label": label,
                "position_label": position_label,
                "ordered_brightness_rank": int(ordered_positions.index(position_label) + 1),
                "pooled_q95": float(bundle_lookup[position_label]["pooled_q95"]),
                "baseline_location": float(bundle_lookup[position_label]["baseline_location"]),
                "baseline_scale": float(bundle_lookup[position_label]["baseline_scale"]),
                "threshold_value": float(bundle_lookup[position_label]["threshold_value"]),
                "early_time_index": int(bundle_lookup[position_label]["early_time"]),
                "early_time_hours": float(bundle_lookup[position_label]["early_time"]) * float(parameters["interval_minutes"]) / 60.0,
                "review_time_indices": ", ".join(str(int(frame["time_index"])) for frame in bundle_lookup[position_label]["review_frames"]),
                "review_times_hours": ", ".join(format_display_hours_from_index(int(frame["time_index"]), 2) for frame in bundle_lookup[position_label]["review_frames"]),
            }
            for selection_rank, (label, position_label) in enumerate(full_example_positions, start=1)
        ]
    )
    threshold_example_selection_path = TABLE_DIR / f"05_{reporter.lower()}_threshold_example_selection.tsv"
    display_time_df(threshold_example_selection_df).to_csv(threshold_example_selection_path, sep="\t", index=False)
    typical_bundle_for_range = full_example_bundles[typical_index]
    if reporter == "RFP" and "Pos36" in bundle_lookup:
        baseline_hist_reference_label = "Pos36"
        baseline_hist_reference_bundle = bundle_lookup[baseline_hist_reference_label]
    else:
        baseline_hist_reference_label = full_example_positions[typical_index][1]
        baseline_hist_reference_bundle = typical_bundle_for_range
    hist_values = typical_bundle_for_range["pooled"]
    hist_xmin = float(np.quantile(hist_values, 0.001))
    hist_xmax = float(np.quantile(hist_values, 0.999))
    hist_xmin = min(hist_xmin, float(typical_bundle_for_range["baseline_location"]), float(typical_bundle_for_range["threshold_value"]))
    hist_xmax = max(hist_xmax, float(typical_bundle_for_range["baseline_location"]), float(typical_bundle_for_range["threshold_value"]))
    if math.isclose(hist_xmax, hist_xmin):
        hist_xmax = hist_xmin + 1.0
    baseline_hist_values = baseline_hist_reference_bundle["baseline_pooled"]
    baseline_hist_xmin = float(np.quantile(baseline_hist_values, 0.001))
    baseline_hist_xmax = float(np.quantile(baseline_hist_values, 0.999))
    baseline_hist_xmin = min(baseline_hist_xmin, float(baseline_hist_reference_bundle["baseline_location"] - baseline_hist_reference_bundle["baseline_scale"]))
    baseline_hist_xmax = max(baseline_hist_xmax, float(baseline_hist_reference_bundle["baseline_location"] + baseline_hist_reference_bundle["baseline_scale"]))
    if math.isclose(baseline_hist_xmax, baseline_hist_xmin):
        baseline_hist_xmax = baseline_hist_xmin + 1.0
    if ((reporter == "RFP" and DEBUG_SHOW_ALL_RFP_THRESHOLD_EXAMPLES) or (reporter == "YFP" and DEBUG_SHOW_ALL_YFP_THRESHOLD_EXAMPLES)):
        display(Markdown(
            f"**{reporter} examples used below**  \nAll `{len(full_example_positions)}` positions are shown below, chunked into debug figures with `{DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE}` rows per figure. Positions are ordered by early pooled `95th percentile corrected intensity`, from dimmer to brighter."
        ))
    else:
        display(Markdown(
            f"**{reporter} examples used below**  \n" + "  \n".join([f"- {label}: `{position_label}`" for label, position_label in full_example_positions])
        ))
    display(Markdown(f"**{reporter} threshold-example selection manifest**  \n`{threshold_example_selection_path.name}`"))
    display(Markdown(
        f"**{reporter} histogram x-range used in all per-position distribution panels below**  \n"
        f"xmin = `{hist_xmin:.0f}`, xmax = `{hist_xmax:.0f}`  \n"
        f"Anchored to the typical early-brightness example: `{full_example_positions[typical_index][1]}`"
    ))
    display(Markdown(
        f"**{reporter} left-half histogram x-range used in the per-position null panels**  \n"
        f"xmin = `{baseline_hist_xmin:.0f}`, xmax = `{baseline_hist_xmax:.0f}`  \n"
        f"Anchored to: `{baseline_hist_reference_label}`"
    ))
    display(Markdown(
        "**Histogram columns**  \n"
        "left histogram = per-position pooled early organoid pixels, with the shared within-cyst baseline peak and shared reporter threshold overlaid; right histogram = only this position's pixels at or below the shared within-cyst baseline peak, with the shared baseline peak and \u00b1 sigma marked"
    ))
    display(Markdown(
        "**Time-course mask strip shown in the rightmost four panels**  \n"
        "underlying image = corrected intensity; cyan = organoid mask, chartreuse = cleaned low-threshold mask after morphology cleanup"
    ))

    for chunk_index, example_positions in enumerate(example_position_chunks, start=1):
        example_bundles = [bundle_lookup[position_label] for _, position_label in example_positions]
        display_vmin, display_vmax = example_corrected_display_limits(example_bundles)
        chunk_position_labels = [position_label for _, position_label in example_positions]
        chunk_header = f"**{reporter} chunk {chunk_index} of {len(example_position_chunks)}**  \n" if len(example_position_chunks) > 1 else f"**{reporter} examples in this figure**  \n"
        display(Markdown(
            chunk_header
            + f"positions: `{', '.join(chunk_position_labels)}`  \n"
            + f"corrected-intensity display range for this figure: `vmin = {display_vmin:.0f}`, `vmax = {display_vmax:.0f}`"
        ))
        n_rows = len(example_positions)
        fig, axes = plt.subplots(n_rows, 8, figsize=(33, max(10.8, 4.6 * n_rows)), constrained_layout=True, squeeze=False)

        for row_index, ((label, position_label), bundle) in enumerate(zip(example_positions, example_bundles)):
            row_label = label.replace("brightness", "brightness").replace("rank", "Rank").capitalize()
            early_corrected_disp = np.clip(bundle["early_corrected"], display_vmin, display_vmax)
            ax = axes[row_index, 0]
            im = ax.imshow(early_corrected_disp, cmap="magma", vmin=display_vmin, vmax=display_vmax)
            draw_mask(ax, bundle["early_mask"], color="deepskyblue", linewidth=1.8)
            if row_index == 0:
                ax.set_title("Early corrected intensity image", fontsize=11, pad=12)
            ax.text(0.03, 0.97, f"{row_label} | {position_label}", transform=ax.transAxes, ha="left", va="top", fontsize=10, color="white", bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.5))
            ax.text(0.03, 0.04, f"t={format_display_hours_from_index(bundle['early_time'], 0)} | whole bg={bundle['early_background']:.0f}", transform=ax.transAxes, ha="left", va="bottom", fontsize=8.3, color="white", bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0))
            ax.axis("off")

            ax = axes[row_index, 1]
            ax.imshow(early_corrected_disp, cmap="magma", vmin=display_vmin, vmax=display_vmax)
            nonbaseline_overlay = np.ma.masked_where(bundle["early_baseline_mask"], np.ones_like(bundle["early_baseline_mask"], dtype=float))
            ax.imshow(nonbaseline_overlay, cmap="gray", alpha=0.72, vmin=0, vmax=1)
            draw_mask(ax, bundle["early_mask"], color="deepskyblue", linewidth=1.8)
            if row_index == 0:
                ax.set_title("Exact pixels at or below the shared within-cyst baseline peak", fontsize=11, pad=12)
            ax.text(0.03, 0.04, f"visible pixels fall on the left-half side of the shared within-cyst baseline peak", transform=ax.transAxes, ha="left", va="bottom", fontsize=8.1, color="white", bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0))
            ax.axis("off")

            ax = axes[row_index, 2]
            pooled = bundle["pooled"]
            ax.hist(pooled, bins=70, color="0.75", edgecolor="white")
            ax.axvline(bundle["baseline_location"], color="deepskyblue", linestyle="--", linewidth=2.0, label="Shared within-cyst baseline peak")
            ax.axvline(bundle["threshold_value"], color="magenta", linewidth=2.4, label="Shared reporter threshold")
            if row_index == 0:
                ax.set_title("Per-position pooled early corrected-intensity distribution", fontsize=11, pad=12)
            ax.text(0.03, 0.97, position_label, transform=ax.transAxes, ha="left", va="top", fontsize=10, fontweight="bold", bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=2.0))
            ax.text(0.03, 0.82, f"q95={bundle['pooled_q95']:.0f} | frac<=peak={bundle['fraction_below_cutoff']:.2f}", transform=ax.transAxes, ha="left", va="top", fontsize=8.2, bbox=dict(facecolor="white", alpha=0.82, edgecolor="none", pad=2.0))
            ax.set_xlim(hist_xmin, hist_xmax)
            if row_index == 0:
                ax.legend(loc="upper right", fontsize=8)
            if row_index == n_rows - 1:
                ax.set_xlabel("Corrected intensity")
            else:
                ax.set_xlabel("")
            ax.set_ylabel("Pixel count")

            ax = axes[row_index, 3]
            baseline_pooled = bundle["baseline_pooled"]
            ax.hist(baseline_pooled, bins=50, color="0.55", edgecolor="white")
            ax.axvline(bundle["baseline_location"], color="deepskyblue", linestyle="--", linewidth=2.0, label="Shared within-cyst baseline peak")
            ax.axvline(bundle["baseline_location"] - bundle["baseline_scale"], color="gold", linestyle=":", linewidth=1.8, label="Peak - sigma")
            ax.axvline(bundle["baseline_location"] + bundle["baseline_scale"], color="gold", linestyle=":", linewidth=1.8, label="Peak + sigma")
            if row_index == 0:
                ax.set_title("Per-position pixels at or below the shared within-cyst baseline peak", fontsize=11, pad=12)
                ax.legend(loc="upper right", fontsize=8)
            ax.text(0.03, 0.97, position_label, transform=ax.transAxes, ha="left", va="top", fontsize=10, fontweight="bold", bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=2.0))
            ax.text(0.03, 0.82, f"global peak={bundle['baseline_location']:.0f} | sigma={bundle['baseline_scale']:.0f}", transform=ax.transAxes, ha="left", va="top", fontsize=8.2, bbox=dict(facecolor="white", alpha=0.82, edgecolor="none", pad=2.0))
            ax.set_xlim(baseline_hist_xmin, baseline_hist_xmax)
            if row_index == n_rows - 1:
                ax.set_xlabel("Corrected intensity")
            else:
                ax.set_xlabel("")
            ax.set_ylabel("Pixel count")

            for review_col, review_frame in enumerate(bundle["review_frames"], start=4):
                ax = axes[row_index, review_col]
                review_corrected_disp = np.clip(review_frame["corrected"], display_vmin, display_vmax)
                ax.imshow(review_corrected_disp, cmap="magma", vmin=display_vmin, vmax=display_vmax)
                draw_mask(ax, review_frame["mask"], color="deepskyblue", linewidth=1.8)
                draw_mask(ax, review_frame["positive_mask"], color="chartreuse", linewidth=1.8)
                if row_index == 0:
                    ax.set_title(f"t={format_display_hours_from_index(review_frame['time_index'], 0)}", fontsize=11, pad=12)
                ax.text(0.03, 0.04, f"t={format_display_hours_from_index(review_frame['time_index'], 0)} | frac={review_frame['positive_fraction']:.3f}", transform=ax.transAxes, ha="left", va="bottom", fontsize=8.0, color="white", bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0))
                ax.axis("off")

        fig_suffix = ""
        title_suffix = ""
        if len(example_position_chunks) > 1:
            fig_suffix = f"_part{chunk_index:02d}"
            start_rank = (chunk_index - 1) * DEBUG_RFP_EXAMPLE_ROWS_PER_FIGURE + 1
            end_rank = start_rank + len(example_positions) - 1
            title_suffix = f" | rows {start_rank}-{end_rank} of {len(full_example_positions)}"
        fig.suptitle(
            f"{reporter} global low-threshold check (fixed corrected-intensity display range across examples shown){title_suffix}",
            fontsize=12.2,
            y=1.002,
        )
        image_axes = [axes[r, c] for r in range(n_rows) for c in [0, 1, 4, 5, 6, 7]]
        fig.colorbar(im, ax=image_axes, fraction=0.02, pad=0.01, label="Corrected intensity (after illumination correction and whole off-cyst subtraction)")
        example_path = FIGURE_DIR / f"05_{reporter.lower()}_threshold_examples{fig_suffix}.png"
        fig.savefig(example_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote threshold example figure:", example_path)
    print("Wrote threshold-example selection table:", threshold_example_selection_path)



## Population Traces

This section summarizes reporter dynamics across all retained positions.

The goal is twofold:

- compare multiple candidate population summaries side by side, including threshold-free alternatives and derivative views
- make unusual positions visible at the population level before dropping into integrated per-position review below


In [ ]:
if DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING:
    retained_frame_metrics = pd.DataFrame()
    population_trace_metrics_df = pd.DataFrame()
    trace_qc_metric_specs = []
    display(Markdown("Skipping population trace plots because `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = True`."))
else:
    retained_frame_metrics = frame_metrics.loc[
        (~frame_metrics["exclude_from_analysis"])
        & (frame_metrics["whole_organoid_metrics_allowed"].fillna(False))
    ].sort_values(["position_label", "time_index", "reporter"]).copy()
    assert_no_excluded_keys_in_analysis(
        frame_metrics,
        retained_frame_metrics,
        key_columns=("position_label", "time_index", "reporter"),
        context="05 retained population-trace frame metrics",
    )

    metric_rows = []
    cached_mask_key = None
    cached_mask = None
    print(
        "Computing derived population-trace metrics for",
        len(retained_frame_metrics),
        "retained reporter frames...",
    )
    for row_index, row in enumerate(retained_frame_metrics.itertuples(index=False), start=1):
        if row_index % 2000 == 0:
            print(f"  processed {row_index}/{len(retained_frame_metrics)} retained reporter frames")

        frame_key = (str(row.position_label), int(row.time_index))
        if frame_key != cached_mask_key:
            cached_mask = load_mask(str(row.position_label), int(row.time_index))
            cached_mask_key = frame_key
        mask = cached_mask
        reporter_image = load_reporter_after_illumination(
            str(row.position_label),
            str(row.reporter),
            int(row.time_index),
        ).astype(float)
        corrected = reporter_image - float(row.background_value)
        organoid_values = corrected[mask]
        baseline_location = float(row.baseline_location)
        baseline_scale = max(abs(float(row.baseline_scale)), 1e-6)
        brightest_decile_mean = brightest_fraction_mean(
            organoid_values,
            POPULATION_BRIGHT_FRACTION,
        )

        record = {
            "position_label": str(row.position_label),
            "time_index": int(row.time_index),
            "time_hours": float(row.time_hours),
            "reporter": str(row.reporter),
            "background_value": float(row.background_value),
            "baseline_location": baseline_location,
            "baseline_scale": baseline_scale,
            "organoid_mean_intensity": float(np.mean(organoid_values)) if organoid_values.size else float("nan"),
            "organoid_mean_intensity_z": (
                float((np.mean(organoid_values) - baseline_location) / baseline_scale)
                if organoid_values.size
                else float("nan")
            ),
            "brightest_decile_mean_intensity": brightest_decile_mean,
            "brightest_decile_mean_intensity_z": (
                float((brightest_decile_mean - baseline_location) / baseline_scale)
                if np.isfinite(brightest_decile_mean)
                else float("nan")
            ),
        }
        for sigma_multiple in POPULATION_THRESHOLD_SIGMAS:
            threshold_value = baseline_location + sigma_multiple * baseline_scale
            positive_mask = positive_mask_from_corrected(
                corrected=corrected,
                organoid_mask=mask,
                threshold_value=threshold_value,
            )
            sigma_suffix = int(sigma_multiple)
            metric_name = f"positive_fraction_sigma{sigma_suffix}"
            positive_pixel_count = int(np.sum(positive_mask))
            record[metric_name] = (
                float(positive_pixel_count / np.sum(mask))
                if np.sum(mask)
                else float("nan")
            )
            positive_values = corrected[positive_mask]
            positive_mean_name = f"positive_mean_intensity_sigma{sigma_suffix}"
            positive_mean_value = (
                float(np.mean(positive_values))
                if positive_values.size
                else float("nan")
            )
            record[positive_mean_name] = positive_mean_value
            record[f"{positive_mean_name}_z"] = (
                float((positive_mean_value - baseline_location) / baseline_scale)
                if np.isfinite(positive_mean_value)
                else float("nan")
            )
        metric_rows.append(record)

    population_trace_metrics_df = pd.DataFrame(metric_rows).sort_values(
        ["position_label", "reporter", "time_index"]
    ).reset_index(drop=True)

    threshold_metric_names = []
    for sigma_multiple in POPULATION_THRESHOLD_SIGMAS:
        sigma_suffix = int(sigma_multiple)
        threshold_metric_names.extend(
            [
                f"positive_fraction_sigma{sigma_suffix}",
                f"positive_mean_intensity_sigma{sigma_suffix}",
                f"positive_mean_intensity_sigma{sigma_suffix}_z",
            ]
        )
    derivative_base_metrics = [
        "organoid_mean_intensity",
        "organoid_mean_intensity_z",
        "brightest_decile_mean_intensity",
        "brightest_decile_mean_intensity_z",
        *threshold_metric_names,
    ]
    for metric_name in derivative_base_metrics:
        population_trace_metrics_df[f"{metric_name}_smooth"] = float("nan")
        population_trace_metrics_df[f"{metric_name}_dt"] = float("nan")
        population_trace_metrics_df[f"{metric_name}_smooth_alt"] = float("nan")
        population_trace_metrics_df[f"{metric_name}_dt_alt"] = float("nan")

    for (position_label, reporter), group in population_trace_metrics_df.groupby(
        ["position_label", "reporter"], sort=True
    ):
        group = group.sort_values("time_hours")
        time_hours = group["time_hours"].to_numpy(dtype=float)
        group_indices = group.index.to_numpy()
        for metric_name in derivative_base_metrics:
            values = group[metric_name].to_numpy(dtype=float)
            smooth_values = rolling_mean_smooth(
                values,
                POPULATION_DERIVATIVE_SMOOTH_WINDOW,
            )
            smooth_values_alt = rolling_mean_smooth(
                values,
                POPULATION_DERIVATIVE_SMOOTH_WINDOW_ALT,
            )
            derivative_values = centered_slope_over_interval(
                smooth_values,
                time_hours,
                POPULATION_DERIVATIVE_INTERVAL_HOURS,
            )
            derivative_values_alt = centered_slope_over_interval(
                smooth_values_alt,
                time_hours,
                POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT,
            )
            population_trace_metrics_df.loc[group_indices, f"{metric_name}_smooth"] = smooth_values
            population_trace_metrics_df.loc[group_indices, f"{metric_name}_dt"] = derivative_values
            population_trace_metrics_df.loc[group_indices, f"{metric_name}_smooth_alt"] = smooth_values_alt
            population_trace_metrics_df.loc[group_indices, f"{metric_name}_dt_alt"] = derivative_values_alt

    population_metrics_path = TABLE_DIR / "05_population_trace_metrics_by_frame.tsv"
    population_trace_metrics_df.to_csv(population_metrics_path, sep="\t", index=False)

    sigma4_check = (
        retained_frame_metrics[
            ["position_label", "reporter", "time_index", "positive_fraction"]
        ]
        .merge(
            population_trace_metrics_df[
                ["position_label", "reporter", "time_index", "positive_fraction_sigma4"]
            ],
            on=["position_label", "reporter", "time_index"],
            how="inner",
        )
    )
    sigma4_abs_diff = np.abs(
        sigma4_check["positive_fraction"].to_numpy(dtype=float)
        - sigma4_check["positive_fraction_sigma4"].to_numpy(dtype=float)
    )
    sigma4_max_abs_diff = float(np.nanmax(sigma4_abs_diff)) if sigma4_abs_diff.size else float("nan")

    population_summary_cache = {}
    summary_metric_names = derivative_base_metrics + [
        "organoid_mean_intensity_z_dt",
        "organoid_mean_intensity_z_dt_alt",
        "brightest_decile_mean_intensity_z_dt",
        "brightest_decile_mean_intensity_z_dt_alt",
    ]
    for sigma_multiple in POPULATION_THRESHOLD_SIGMAS:
        sigma_suffix = int(sigma_multiple)
        summary_metric_names.extend(
            [
                f"positive_fraction_sigma{sigma_suffix}_dt",
                f"positive_fraction_sigma{sigma_suffix}_dt_alt",
                f"positive_mean_intensity_sigma{sigma_suffix}_z_dt",
                f"positive_mean_intensity_sigma{sigma_suffix}_z_dt_alt",
            ]
        )
    for metric_name in summary_metric_names:
        population_summary_cache[metric_name] = summarize_population_by_time(
            population_trace_metrics_df[["reporter", "time_hours", metric_name]].copy(),
            metric_name,
        )

    trace_qc_metric_specs = [
        {
            "metric_name": "organoid_mean_intensity_z",
            "metric_title": "Whole-cyst mean reporter intensity (sigma above pooled early within-cyst baseline)",
            "metric_short": "whole-cyst mean intensity (z)",
        },
        {
            "metric_name": "brightest_decile_mean_intensity_z",
            "metric_title": "Brightest 10% of cyst pixels: mean intensity (sigma above pooled early within-cyst baseline)",
            "metric_short": "brightest 10% mean intensity (z)",
        },
        {
            "metric_name": "positive_fraction_sigma3",
            "metric_title": "Positive fraction within cyst mask (threshold = shared within-cyst baseline + 3 sigma)",
            "metric_short": "positive fraction at 3 sigma",
        },
        {
            "metric_name": "positive_fraction_sigma4",
            "metric_title": "Positive fraction within cyst mask (threshold = shared within-cyst baseline + 4 sigma)",
            "metric_short": "positive fraction at 4 sigma",
        },
    ]

    def render_population_grid(
        metric_specs: list[dict[str, object]],
        figure_path: Path,
        figure_title: str,
        yscale: str = "linear",
        sharey_mode: str | bool = "col",
        line_stat: str = "mean",
        show_individual: bool = True,
        show_band: bool = False,
    ) -> None:
        n_cols = len(metric_specs)
        fig, axes = plt.subplots(
            2,
            n_cols,
            figsize=(4.55 * n_cols, 6.9),
            sharex=True,
            sharey=sharey_mode,
            constrained_layout=True,
        )
        if n_cols == 1:
            axes = np.asarray(axes).reshape(2, 1)
        for row_index, reporter in enumerate(["RFP", "YFP"]):
            reporter_df = population_trace_metrics_df.loc[
                population_trace_metrics_df["reporter"] == reporter
            ].copy()
            for col_index, spec in enumerate(metric_specs):
                ax = axes[row_index, col_index]
                metric_name = str(spec["metric_name"])
                summary_df = population_summary_cache[metric_name].loc[
                    population_summary_cache[metric_name]["reporter"] == reporter
                ].sort_values("time_hours")
                if show_individual:
                    for _, sub in reporter_df.groupby("position_label", sort=False):
                        sub = sub.sort_values("time_hours")
                        ax.plot(
                            sub["time_hours"],
                            sub[metric_name],
                            color=REPORTER_INDIVIDUAL_COLORS[reporter],
                            alpha=0.10,
                            linewidth=0.8,
                            linestyle=POPULATION_REPORTER_LINESTYLES[reporter],
                            zorder=1,
                        )
                if show_band and {"q25", "q75"}.issubset(summary_df.columns):
                    ax.fill_between(
                        summary_df["time_hours"],
                        summary_df["q25"],
                        summary_df["q75"],
                        color=REPORTER_COLORS[reporter],
                        alpha=0.16,
                        zorder=2,
                    )
                ax.plot(
                    summary_df["time_hours"],
                    summary_df[line_stat],
                    color=REPORTER_COLORS[reporter],
                    linewidth=2.5,
                    linestyle=POPULATION_REPORTER_LINESTYLES[reporter],
                    zorder=3 if show_band else 2,
                )
                if row_index == 0:
                    ax.set_title(str(spec["title"]), fontsize=9.3)
                if col_index == 0:
                    ax.set_ylabel(f"{reporter}\n{spec['ylabel']}", fontsize=9.5)
                if row_index == 1:
                    ax.set_xlabel("Time (hours)")
                    set_display_time_axis(ax, "x")
                    set_display_time_axis(ax, "x")
                    set_display_time_axis(ax, "x")
                if spec.get("ylim_by_reporter") is not None:
                    reporter_ylim = spec["ylim_by_reporter"].get(reporter)
                    if reporter_ylim is not None:
                        ax.set_ylim(reporter_ylim)
                if spec.get("ylim") is not None:
                    ax.set_ylim(spec["ylim"])
                if yscale == "symlog":
                    ax.set_yscale(
                        "symlog",
                        linthresh=float(spec.get("linthresh", 0.1)),
                    )
                ax.grid(alpha=0.18)
        fig.suptitle(figure_title, fontsize=12.2, y=1.02)
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)


    def render_single_reporter_population_grid(
        metric_specs: list[dict[str, object]],
        reporter: str,
        figure_path: Path,
        figure_title: str,
        yscale: str = "linear",
        sharey: bool = False,
        line_stat: str = "mean",
        show_individual: bool = True,
        show_band: bool = False,
    ) -> None:
        n_cols = len(metric_specs)
        fig, axes = plt.subplots(
            1,
            n_cols,
            figsize=(4.55 * n_cols, 3.55),
            sharex=True,
            sharey=sharey,
            constrained_layout=True,
            squeeze=False,
        )
        axes = axes[0]
        reporter_df = population_trace_metrics_df.loc[
            population_trace_metrics_df["reporter"] == reporter
        ].copy()
        for col_index, spec in enumerate(metric_specs):
            ax = axes[col_index]
            metric_name = str(spec["metric_name"])
            summary_df = population_summary_cache[metric_name].loc[
                population_summary_cache[metric_name]["reporter"] == reporter
            ].sort_values("time_hours")
            if show_individual:
                for _, sub in reporter_df.groupby("position_label", sort=False):
                    sub = sub.sort_values("time_hours")
                    ax.plot(
                        sub["time_hours"],
                        sub[metric_name],
                        color=REPORTER_INDIVIDUAL_COLORS[reporter],
                        alpha=0.10,
                        linewidth=0.8,
                        linestyle=POPULATION_REPORTER_LINESTYLES[reporter],
                        zorder=1,
                    )
            if show_band and {"q25", "q75"}.issubset(summary_df.columns):
                ax.fill_between(
                    summary_df["time_hours"],
                    summary_df["q25"],
                    summary_df["q75"],
                    color=REPORTER_COLORS[reporter],
                    alpha=0.16,
                    zorder=2,
                )
            ax.plot(
                summary_df["time_hours"],
                summary_df[line_stat],
                color=REPORTER_COLORS[reporter],
                linewidth=2.6,
                linestyle=POPULATION_REPORTER_LINESTYLES[reporter],
                zorder=3 if show_band else 2,
            )
            ax.set_title(str(spec["title"]), fontsize=9.5)
            if col_index == 0:
                ax.set_ylabel(str(spec["ylabel"]), fontsize=9.6)
            ax.set_xlabel("Time (hours)")
            set_display_time_axis(ax, "x")
            if spec.get("ylim_by_reporter") is not None:
                reporter_ylim = spec["ylim_by_reporter"].get(reporter)
                if reporter_ylim is not None:
                    ax.set_ylim(reporter_ylim)
            if spec.get("ylim") is not None:
                ax.set_ylim(spec["ylim"])
            if yscale == "symlog":
                ax.set_yscale(
                    "symlog",
                    linthresh=float(spec.get("linthresh", 0.1)),
                )
            ax.grid(alpha=0.18)
        fig.suptitle(f"{reporter}: {figure_title}", fontsize=12.0, y=1.03)
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)


    def choose_threshold_demo_time(
        position_label: str,
        reporter: str,
        target_sigma_multiple: int = 3,
        target_fraction: float = 0.22,
    ) -> int:
        metric_name = f"positive_fraction_sigma{int(target_sigma_multiple)}"
        subset = population_trace_metrics_df.loc[
            (population_trace_metrics_df["position_label"] == position_label)
            & (population_trace_metrics_df["reporter"] == reporter)
        ].sort_values("time_index").copy()
        if subset.empty:
            raise RuntimeError(f"No retained frames available for {position_label} {reporter}")
        finite = subset.loc[np.isfinite(subset[metric_name])].copy()
        if finite.empty:
            return int(subset["time_index"].iloc[len(subset) // 2])
        positive = finite.loc[finite[metric_name] > 0].copy()
        if not positive.empty:
            positive["score"] = np.abs(
                positive[metric_name].to_numpy(dtype=float) - float(target_fraction)
            )
            positive = positive.sort_values(
                ["score", metric_name, "time_index"],
                ascending=[True, False, True],
            )
            return int(positive["time_index"].iloc[0])
        max_index = finite[metric_name].astype(float).idxmax()
        return int(finite.loc[max_index, "time_index"])


    def threshold_choice_demo_bundle(
        position_label: str,
        reporter: str,
        sigma_multiples: list[int],
        target_sigma_multiple: int = 3,
        target_fraction: float = 0.22,
    ) -> dict[str, object]:
        threshold_row = threshold_df.loc[
            (threshold_df["position_label"] == position_label)
            & (threshold_df["reporter"] == reporter)
        ].iloc[0]
        baseline_location = float(threshold_row["baseline_location"])
        baseline_scale = float(threshold_row["baseline_scale"])
        time_index = choose_threshold_demo_time(
            position_label,
            reporter,
            target_sigma_multiple=target_sigma_multiple,
            target_fraction=target_fraction,
        )
        mask = load_mask(position_label, time_index)
        reporter_image = load_reporter_after_illumination(
            position_label,
            reporter,
            time_index,
        ).astype(float)
        corrected, background_value, _ = corrected_signal(
            signal=reporter_image,
            organoid_mask=mask,
            background_estimator=str(parameters.get("background_estimator", "whole_off_cyst")),
            background_ring_inner=int(parameters["background_ring_inner"]),
            background_ring_outer=int(parameters["background_ring_outer"]),
            min_ring_pixels=int(parameters["min_ring_pixels"]),
        )
        positive_masks = {}
        positive_fractions = {}
        threshold_values = {}
        for sigma_multiple in sigma_multiples:
            threshold_value = baseline_location + float(sigma_multiple) * baseline_scale
            threshold_values[int(sigma_multiple)] = float(threshold_value)
            positive_mask = positive_mask_from_corrected(
                corrected=corrected,
                organoid_mask=mask,
                threshold_value=threshold_value,
            )
            positive_masks[int(sigma_multiple)] = positive_mask
            positive_fractions[int(sigma_multiple)] = (
                float(np.sum(positive_mask) / np.sum(mask))
                if np.sum(mask)
                else float("nan")
            )
        return {
            "position_label": position_label,
            "reporter": reporter,
            "time_index": int(time_index),
            "corrected": corrected,
            "mask": mask,
            "background_value": float(background_value),
            "baseline_location": baseline_location,
            "baseline_scale": baseline_scale,
            "threshold_values": threshold_values,
            "positive_masks": positive_masks,
            "positive_fractions": positive_fractions,
        }


    def render_threshold_choice_demo(
        reporter: str,
        figure_path: Path,
        sigma_multiples: list[int],
    ) -> None:
        demo_example_count = 10
        example_positions = []
        seen_positions = set()
        example_lookup = threshold_examples.get(reporter, {})
        ordered_positions = list(example_lookup.get("ordered_positions", []))
        if ordered_positions:
            raw_indices = np.linspace(
                0,
                len(ordered_positions) - 1,
                num=min(demo_example_count, len(ordered_positions)),
            ).round().astype(int)
            for idx in raw_indices:
                idx = max(0, min(len(ordered_positions) - 1, int(idx)))
                position_label = ordered_positions[idx]
                if position_label in seen_positions:
                    continue
                example_positions.append(
                    (f"brightness rank {idx + 1} of {len(ordered_positions)}", position_label)
                )
                seen_positions.add(position_label)
        if len(example_positions) < min(demo_example_count, len(ordered_positions) if ordered_positions else demo_example_count):
            if not ordered_positions:
                ordered_positions = (
                    threshold_df.loc[threshold_df["reporter"] == reporter, "position_label"]
                    .drop_duplicates()
                    .tolist()
                )
            for position_label in ordered_positions:
                if position_label in seen_positions:
                    continue
                example_positions.append(
                    (f"brightness rank {len(example_positions) + 1}", position_label)
                )
                seen_positions.add(position_label)
                if len(example_positions) >= demo_example_count:
                    break

        bundles = [
            threshold_choice_demo_bundle(position_label, reporter, sigma_multiples)
            for _, position_label in example_positions
        ]
        selection_df = pd.DataFrame(
            [
                {
                    "selection_rank": int(row_index + 1),
                    "display_label": str(display_label),
                    "position_label": str(position_label),
                    "time_index": int(bundle["time_index"]),
                    "time_hours": float(bundle["time_index"]) * float(parameters["interval_minutes"]) / 60.0,
                    "background_value": float(bundle["background_value"]),
                    **{
                        f"positive_fraction_sigma{int(sigma_multiple)}": float(bundle["positive_fractions"][int(sigma_multiple)])
                        for sigma_multiple in sigma_multiples
                    },
                }
                for row_index, ((display_label, position_label), bundle) in enumerate(zip(example_positions, bundles))
            ]
        )
        selection_path = TABLE_DIR / f"05_population_positive_fraction_threshold_examples_{reporter.lower()}_selection.tsv"
        display_time_df(selection_df).to_csv(selection_path, sep="\t", index=False)
        display(Markdown(f"**{reporter} population threshold-demo selection manifest**  \n`{selection_path.name}`"))
        display_vmin, display_vmax = corrected_display_limits_from_arrays(
            [bundle["corrected"] for bundle in bundles]
        )
        n_rows = len(bundles)
        n_cols = 1 + len(sigma_multiples)
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(4.7 * n_cols, max(7.2, 3.7 * n_rows)),
            constrained_layout=True,
            squeeze=False,
        )

        for row_index, ((display_label, position_label), bundle) in enumerate(zip(example_positions, bundles)):
            corrected_disp = np.clip(bundle["corrected"], display_vmin, display_vmax)
            ax = axes[row_index, 0]
            im = ax.imshow(corrected_disp, cmap="magma", vmin=display_vmin, vmax=display_vmax)
            draw_mask(ax, bundle["mask"], color="deepskyblue", linewidth=1.8)
            if row_index == 0:
                ax.set_title("Representative corrected image", fontsize=10.8, pad=10)
            ax.text(
                0.03,
                0.97,
                f"{display_label} | {position_label}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=9.4,
                color="white",
                bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.4),
            )
            ax.text(
                0.03,
                0.04,
                f"t={format_display_hours_from_index(bundle['time_index'], 0)} | bg={bundle['background_value']:.0f}",
                transform=ax.transAxes,
                ha="left",
                va="bottom",
                fontsize=8.1,
                color="white",
                bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0),
            )
            ax.axis("off")

            for col_index, sigma_multiple in enumerate(sigma_multiples, start=1):
                ax = axes[row_index, col_index]
                ax.imshow(corrected_disp, cmap="magma", vmin=display_vmin, vmax=display_vmax)
                draw_mask(ax, bundle["mask"], color="deepskyblue", linewidth=1.8)
                draw_mask(ax, bundle["positive_masks"][int(sigma_multiple)], color="chartreuse", linewidth=1.8)
                if row_index == 0:
                    ax.set_title(
                        f"{int(sigma_multiple)} sigma threshold",
                        fontsize=10.8,
                        pad=10,
                    )
                ax.text(
                    0.03,
                    0.04,
                    f"frac={bundle['positive_fractions'][int(sigma_multiple)]:.3f}",
                    transform=ax.transAxes,
                    ha="left",
                    va="bottom",
                    fontsize=8.0,
                    color="white",
                    bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0),
                )
                ax.axis("off")

        fig.suptitle(
            f"{reporter}: what the threshold choices look like on representative corrected images",
            fontsize=12.0,
            y=1.01,
        )
        fig.colorbar(
            im,
            ax=[axes[r, c] for r in range(n_rows) for c in range(n_cols)],
            fraction=0.02,
            pad=0.01,
            label="Corrected intensity (after illumination correction and whole off-cyst subtraction)",
        )
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)
        print("Wrote selection table:", selection_path)


    def apply_named_smoother(
        values: np.ndarray,
        method_name: str,
        smooth_window: int,
    ) -> np.ndarray:
        if method_name == "rolling mean":
            return rolling_mean_smooth(values, smooth_window)
        if method_name == "rolling median":
            return rolling_median_smooth(values, smooth_window)
        if method_name == "gaussian":
            return gaussian_smooth(values, max(1.0, smooth_window / 3.0))
        raise ValueError(f"Unknown smoother: {method_name}")


    def summarize_mean_of_position_derivatives(
        metric_name: str,
        reporter: str,
        smooth_window: int,
        interval_hours: float,
        method_name: str = "rolling mean",
    ) -> pd.DataFrame:
        rows = []
        reporter_df = population_trace_metrics_df.loc[
            population_trace_metrics_df["reporter"] == reporter,
            ["position_label", "time_hours", metric_name],
        ].copy()
        for _, sub in reporter_df.groupby("position_label", sort=True):
            sub = sub.sort_values("time_hours")
            times = sub["time_hours"].to_numpy(dtype=float)
            values = sub[metric_name].to_numpy(dtype=float)
            smoothed = apply_named_smoother(values, method_name, smooth_window)
            deriv = centered_slope_over_interval(smoothed, times, interval_hours)
            rows.append(
                pd.DataFrame(
                    {
                        "time_hours": times,
                        "derivative": deriv,
                    }
                )
            )
        if not rows:
            return pd.DataFrame(columns=["time_hours", "mean"])
        derivative_df = pd.concat(rows, ignore_index=True)
        derivative_df = derivative_df.loc[np.isfinite(derivative_df["derivative"])]
        if derivative_df.empty:
            return pd.DataFrame(columns=["time_hours", "mean"])
        return (
            derivative_df.groupby("time_hours", as_index=False)["derivative"]
            .mean()
            .rename(columns={"derivative": "mean"})
            .sort_values("time_hours")
            .reset_index(drop=True)
        )


    def summarize_derivative_of_aggregate_mean(
        metric_name: str,
        reporter: str,
        smooth_window: int,
        interval_hours: float,
        method_name: str = "rolling mean",
    ) -> pd.DataFrame:
        summary_df = population_summary_cache[metric_name].loc[
            population_summary_cache[metric_name]["reporter"] == reporter
        ].sort_values("time_hours")
        times = summary_df["time_hours"].to_numpy(dtype=float)
        mean_values = summary_df["mean"].to_numpy(dtype=float)
        smoothed = apply_named_smoother(mean_values, method_name, smooth_window)
        deriv = centered_slope_over_interval(smoothed, times, interval_hours)
        return pd.DataFrame({"time_hours": times, "mean": deriv}).sort_values("time_hours").reset_index(drop=True)


    def render_derivative_comparison_grid(
        metric_specs: list[dict[str, object]],
        figure_path: Path,
        figure_title: str,
        parameter_choices: list[dict[str, object]],
    ) -> None:
        n_cols = len(metric_specs)
        fig, axes = plt.subplots(
            2,
            n_cols,
            figsize=(4.8 * n_cols, 6.8),
            sharex=True,
            constrained_layout=True,
        )
        if n_cols == 1:
            axes = np.asarray(axes).reshape(2, 1)
        linestyle_by_form = {
            "Derivative of aggregate mean": "-",
            "Mean of per-position derivatives": "--",
        }
        for row_index, reporter in enumerate(["RFP", "YFP"]):
            for col_index, spec in enumerate(metric_specs):
                ax = axes[row_index, col_index]
                metric_name = str(spec["metric_name"])
                plotted_arrays = []
                for choice in parameter_choices:
                    color = str(choice["color"])
                    window = int(choice["window"])
                    interval_hours = float(choice["interval_hours"])
                    method_name = str(choice.get("method_name", "rolling mean"))
                    label_stub = f"{window}f + {interval_hours:g}h"

                    aggregate_deriv = summarize_derivative_of_aggregate_mean(
                        metric_name,
                        reporter,
                        smooth_window=window,
                        interval_hours=interval_hours,
                        method_name=method_name,
                    )
                    position_mean_deriv = summarize_mean_of_position_derivatives(
                        metric_name,
                        reporter,
                        smooth_window=window,
                        interval_hours=interval_hours,
                        method_name=method_name,
                    )
                    for label, derivative_df in [
                        ("Derivative of aggregate mean", aggregate_deriv),
                        ("Mean of per-position derivatives", position_mean_deriv),
                    ]:
                        derivative_df = derivative_df.loc[np.isfinite(derivative_df["mean"])].copy()
                        if derivative_df.empty:
                            continue
                        plotted_arrays.append(derivative_df["mean"].to_numpy(dtype=float))
                        legend_label = f"{label_stub} | {label}"
                        ax.plot(
                            derivative_df["time_hours"],
                            derivative_df["mean"],
                            color=color,
                            linewidth=2.0,
                            linestyle=linestyle_by_form[label],
                            label=legend_label,
                        )
                ax.axhline(0.0, color="0.65", linewidth=1.0, linestyle=":")
                if plotted_arrays:
                    values = np.concatenate(plotted_arrays)
                    values = values[np.isfinite(values)]
                    if values.size:
                        lower = float(np.min(values))
                        upper = float(np.max(values))
                        span = upper - lower
                        if not np.isfinite(span) or span <= 0:
                            span = max(abs(upper), 1.0)
                        pad = max(1e-6, 0.10 * span)
                        ax.set_ylim(lower - pad, upper + pad)
                if row_index == 0:
                    ax.set_title(str(spec["title"]), fontsize=9.5)
                if col_index == 0:
                    ax.set_ylabel(f"{reporter}\n{spec['ylabel']}", fontsize=9.5)
                if row_index == 1:
                    ax.set_xlabel("Time (hours)")
                    set_display_time_axis(ax, "x")
                if row_index == 0 and col_index == n_cols - 1:
                    ax.legend(loc="upper left", fontsize=7.4, frameon=False)
                ax.grid(alpha=0.18)
        fig.suptitle(figure_title, fontsize=12.1, y=1.02)
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)


    def render_aggregate_derivative_window_sweep(
        metric_specs: list[dict[str, object]],
        figure_path: Path,
        figure_title: str,
    ) -> None:
        window_colors = {11: "#4c72b0", 21: "#55a868", 41: "#c44e52"}
        n_cols = len(metric_specs)
        fig, axes = plt.subplots(
            2,
            n_cols,
            figsize=(4.55 * n_cols, 6.6),
            sharex=True,
            constrained_layout=True,
        )
        if n_cols == 1:
            axes = np.asarray(axes).reshape(2, 1)
        for row_index, reporter in enumerate(["RFP", "YFP"]):
            for col_index, spec in enumerate(metric_specs):
                ax = axes[row_index, col_index]
                summary_df = population_summary_cache[str(spec["metric_name"])].loc[
                    population_summary_cache[str(spec["metric_name"])]["reporter"] == reporter
                ].sort_values("time_hours")
                time_hours = summary_df["time_hours"].to_numpy(dtype=float)
                mean_values = summary_df["mean"].to_numpy(dtype=float)
                for window in POPULATION_DERIVATIVE_WINDOW_SWEEP:
                    smoothed = rolling_mean_smooth(mean_values, window)
                    deriv = centered_slope_over_interval(
                        smoothed,
                        time_hours,
                        POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT,
                    )
                    ax.plot(
                        time_hours,
                        deriv,
                        color=window_colors.get(window, "0.3"),
                        linewidth=2.0,
                        label=f"{window}-frame mean + {POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT:g}-hour slope",
                    )
                ax.axhline(0.0, color="0.6", linewidth=1.0, linestyle=":")
                if row_index == 0:
                    ax.set_title(str(spec["title"]), fontsize=9.3)
                if col_index == 0:
                    ax.set_ylabel(f"{reporter}\n{spec['ylabel']}", fontsize=9.5)
                if row_index == 1:
                    ax.set_xlabel("Time (hours)")
                    set_display_time_axis(ax, "x")
                if row_index == 0 and col_index == n_cols - 1:
                    ax.legend(loc="upper left", fontsize=8, frameon=False)
                ax.grid(alpha=0.18)
        fig.suptitle(figure_title, fontsize=12.2, y=1.02)
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)


    def render_aggregate_derivative_interval_sweep(
        metric_specs: list[dict[str, object]],
        figure_path: Path,
        figure_title: str,
    ) -> None:
        interval_colors = {2.0: "#4c72b0", 4.0: "#55a868", 8.0: "#c44e52"}
        n_cols = len(metric_specs)
        fig, axes = plt.subplots(
            2,
            n_cols,
            figsize=(4.55 * n_cols, 6.6),
            sharex=True,
            constrained_layout=True,
        )
        if n_cols == 1:
            axes = np.asarray(axes).reshape(2, 1)
        for row_index, reporter in enumerate(["RFP", "YFP"]):
            for col_index, spec in enumerate(metric_specs):
                ax = axes[row_index, col_index]
                summary_df = population_summary_cache[str(spec["metric_name"])].loc[
                    population_summary_cache[str(spec["metric_name"])]["reporter"] == reporter
                ].sort_values("time_hours")
                time_hours = summary_df["time_hours"].to_numpy(dtype=float)
                mean_values = summary_df["mean"].to_numpy(dtype=float)
                smoothed = rolling_mean_smooth(
                    mean_values,
                    POPULATION_DERIVATIVE_SMOOTH_WINDOW_ALT,
                )
                for interval_hours in POPULATION_DERIVATIVE_INTERVAL_SWEEP_HOURS:
                    deriv = centered_slope_over_interval(
                        smoothed,
                        time_hours,
                        interval_hours,
                    )
                    ax.plot(
                        time_hours,
                        deriv,
                        color=interval_colors.get(interval_hours, "0.3"),
                        linewidth=2.0,
                        label=f"{interval_hours:g}-hour centered slope",
                    )
                ax.axhline(0.0, color="0.6", linewidth=1.0, linestyle=":")
                if row_index == 0:
                    ax.set_title(str(spec["title"]), fontsize=9.3)
                if col_index == 0:
                    ax.set_ylabel(f"{reporter}\n{spec['ylabel']}", fontsize=9.5)
                if row_index == 1:
                    ax.set_xlabel("Time (hours)")
                    set_display_time_axis(ax, "x")
                if row_index == 0 and col_index == n_cols - 1:
                    ax.legend(loc="upper left", fontsize=8, frameon=False)
                ax.grid(alpha=0.18)
        fig.suptitle(figure_title, fontsize=12.2, y=1.02)
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)


    def render_aggregate_derivative_method_sweep(
        metric_specs: list[dict[str, object]],
        figure_path: Path,
        figure_title: str,
    ) -> None:
        method_specs = [
            ("rolling mean", "#4c72b0"),
            ("rolling median", "#55a868"),
            ("gaussian", "#c44e52"),
        ]
        n_cols = len(metric_specs)
        fig, axes = plt.subplots(
            2,
            n_cols,
            figsize=(4.55 * n_cols, 6.6),
            sharex=True,
            constrained_layout=True,
        )
        if n_cols == 1:
            axes = np.asarray(axes).reshape(2, 1)
        for row_index, reporter in enumerate(["RFP", "YFP"]):
            for col_index, spec in enumerate(metric_specs):
                ax = axes[row_index, col_index]
                summary_df = population_summary_cache[str(spec["metric_name"])].loc[
                    population_summary_cache[str(spec["metric_name"])]["reporter"] == reporter
                ].sort_values("time_hours")
                time_hours = summary_df["time_hours"].to_numpy(dtype=float)
                mean_values = summary_df["mean"].to_numpy(dtype=float)
                smoothed_map = {
                    "rolling mean": rolling_mean_smooth(
                        mean_values,
                        POPULATION_DERIVATIVE_SMOOTH_WINDOW_ALT,
                    ),
                    "rolling median": rolling_median_smooth(
                        mean_values,
                        POPULATION_DERIVATIVE_SMOOTH_WINDOW_ALT,
                    ),
                    "gaussian": gaussian_smooth(
                        mean_values,
                        POPULATION_DERIVATIVE_GAUSSIAN_SIGMA_FRAMES,
                    ),
                }
                for method_name, color in method_specs:
                    deriv = centered_slope_over_interval(
                        smoothed_map[method_name],
                        time_hours,
                        POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT,
                    )
                    ax.plot(
                        time_hours,
                        deriv,
                        color=color,
                        linewidth=2.0,
                        label=f"{method_name}, {POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT:g}-hour slope",
                    )
                ax.axhline(0.0, color="0.6", linewidth=1.0, linestyle=":")
                if row_index == 0:
                    ax.set_title(str(spec["title"]), fontsize=9.3)
                if col_index == 0:
                    ax.set_ylabel(f"{reporter}\n{spec['ylabel']}", fontsize=9.5)
                if row_index == 1:
                    ax.set_xlabel("Time (hours)")
                if row_index == 0 and col_index == n_cols - 1:
                    ax.legend(loc="upper left", fontsize=8, frameon=False)
                ax.grid(alpha=0.18)
        fig.suptitle(figure_title, fontsize=12.2, y=1.02)
        fig.savefig(figure_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)
        print("Wrote population figure:", figure_path)

    display(
        Markdown(
            f'''
            **Population trace plotting families**

            All population plots below use only retained frames with `whole_organoid_metrics_allowed = True`.

            - pale rose lines = individual `RFP` positions
            - pale gold lines = individual `YFP` positions
            - dark reporter-colored lines = mean across retained positions
            - threshold-sweep fractions are recomputed directly from corrected images with the same morphology cleanup used in the main segmentation
            - threshold-sweep positive-region mean intensity is measured only within the cleaned positive mask for that threshold; it is undefined (`NaN`) when no pixels survive thresholding and cleanup
            - the standardized threshold-free plots use the exact same cyst-level summaries as the corrected-intensity plots, but rescaled within each reporter as `(summary - pooled early within-cyst baseline) / within-cyst sigma`
            - derivative plots below are aggregate-only comparison plots: no individual traces and no spread bands
            - each derivative panel compares `derivative of aggregate mean` versus `mean of per-position derivatives` under several linear smoothing / interval choices
            - the threshold-free y-axis limits are intentionally focused on the aggregate behavior, so some outlier individual traces may clip
            - because corrected intensities and derivatives can cross zero, the log-style alternatives below use `symlog` rather than pure log scaling

            Wrote derived per-frame population metrics: `{population_metrics_path.name}`

            Sanity check against the main `05` positive fraction:
            - `positive_fraction_sigma4` vs current `positive_fraction`: max absolute difference = `{sigma4_max_abs_diff:.3g}`
            '''
        )
    )

    positive_fraction_specs = [
        {
            "metric_name": f"positive_fraction_sigma{int(sigma_multiple)}",
            "title": f"{int(sigma_multiple)} sigma threshold",
            "ylabel": "fraction of cyst mask\nafter morphology cleanup",
            "ylim": (-0.02, 1.02),
        }
        for sigma_multiple in POPULATION_THRESHOLD_SIGMAS
    ]

    threshold_free_linear_ylims = {}
    for metric_name in [
        "organoid_mean_intensity",
        "brightest_decile_mean_intensity",
        "organoid_mean_intensity_z",
        "brightest_decile_mean_intensity_z",
    ]:
        threshold_free_linear_ylims[metric_name] = {}
        for reporter in ["RFP", "YFP"]:
            reporter_summary = population_summary_cache[metric_name].loc[
                population_summary_cache[metric_name]["reporter"] == reporter
            ].copy()
            padding_fraction = 0.10 if metric_name.endswith("_z") else 0.12
            threshold_free_linear_ylims[metric_name][reporter] = focus_ylim_from_summary(
                reporter_summary,
                include_zero=False,
                padding_fraction=padding_fraction,
            )

    threshold_free_symlog_ylims = {}
    for metric_name in [
        "organoid_mean_intensity",
        "brightest_decile_mean_intensity",
        "organoid_mean_intensity_z",
        "brightest_decile_mean_intensity_z",
    ]:
        threshold_free_symlog_ylims[metric_name] = {}
        for reporter in ["RFP", "YFP"]:
            reporter_summary = population_summary_cache[metric_name].loc[
                population_summary_cache[metric_name]["reporter"] == reporter
            ].copy()
            threshold_free_symlog_ylims[metric_name][reporter] = focus_ylim_from_summary(
                reporter_summary,
                include_zero=True,
                padding_fraction=0.08,
            )

    positive_mean_linear_ylims = {}
    positive_mean_metric_names = []
    for sigma_multiple in POPULATION_THRESHOLD_SIGMAS:
        sigma_suffix = int(sigma_multiple)
        positive_mean_metric_names.extend(
            [
                f"positive_mean_intensity_sigma{sigma_suffix}",
                f"positive_mean_intensity_sigma{sigma_suffix}_z",
            ]
        )
    for metric_name in positive_mean_metric_names:
        positive_mean_linear_ylims[metric_name] = {}
        for reporter in ["RFP", "YFP"]:
            reporter_summary = population_summary_cache[metric_name].loc[
                population_summary_cache[metric_name]["reporter"] == reporter
            ].copy()
            positive_mean_linear_ylims[metric_name][reporter] = focus_ylim_from_summary(
                reporter_summary,
                include_zero=False,
                padding_fraction=0.06 if metric_name.endswith("_z") else 0.12,
            )

    positive_mean_specs = [
        {
            "metric_name": f"positive_mean_intensity_sigma{int(sigma_multiple)}",
            "title": f"{int(sigma_multiple)} sigma threshold",
            "ylabel": "corrected intensity\nwithin cleaned positive mask",
            "ylim_by_reporter": positive_mean_linear_ylims[
                f"positive_mean_intensity_sigma{int(sigma_multiple)}"
            ],
        }
        for sigma_multiple in POPULATION_THRESHOLD_SIGMAS
    ]

    positive_mean_standardized_specs = [
        {
            "metric_name": f"positive_mean_intensity_sigma{int(sigma_multiple)}_z",
            "title": f"{int(sigma_multiple)} sigma threshold",
            "ylabel": "sigma above\npooled early within-cyst baseline",
            "ylim_by_reporter": positive_mean_linear_ylims[
                f"positive_mean_intensity_sigma{int(sigma_multiple)}_z"
            ],
        }
        for sigma_multiple in POPULATION_THRESHOLD_SIGMAS
    ]

    corrected_intensity_specs = [
        {
            "metric_name": "organoid_mean_intensity",
            "title": "Whole-cyst mean corrected intensity",
            "ylabel": "corrected intensity\nwithin full cyst mask",
            "ylim_by_reporter": threshold_free_linear_ylims["organoid_mean_intensity"],
        },
        {
            "metric_name": "brightest_decile_mean_intensity",
            "title": "Brightest 10% of cyst pixels: mean corrected intensity",
            "ylabel": "corrected intensity\nbrightest 10% of cyst pixels",
            "ylim_by_reporter": threshold_free_linear_ylims["brightest_decile_mean_intensity"],
        },
    ]

    corrected_symlog_specs = [
        {
            **spec,
            "ylim_by_reporter": threshold_free_symlog_ylims[spec["metric_name"]],
            "linthresh": symlog_linthresh(
                population_trace_metrics_df[spec["metric_name"]].to_numpy(dtype=float),
                min_linthresh=25.0,
            ),
        }
        for spec in corrected_intensity_specs
    ]

    standardized_intensity_specs = [
        {
            "metric_name": "organoid_mean_intensity_z",
            "title": "Whole-cyst mean intensity standardized to pooled early within-cyst baseline",
            "ylabel": "sigma above\npooled early within-cyst baseline",
            "ylim_by_reporter": threshold_free_linear_ylims["organoid_mean_intensity_z"],
        },
        {
            "metric_name": "brightest_decile_mean_intensity_z",
            "title": "Brightest 10% of cyst pixels: standardized mean intensity",
            "ylabel": "sigma above\npooled early within-cyst baseline",
            "ylim_by_reporter": threshold_free_linear_ylims["brightest_decile_mean_intensity_z"],
        },
    ]
    derivative_parameter_choices = [
        {
            "window": POPULATION_DERIVATIVE_SMOOTH_WINDOW,
            "interval_hours": POPULATION_DERIVATIVE_INTERVAL_HOURS,
            "color": "#4c72b0",
            "method_name": "rolling mean",
        },
        {
            "window": POPULATION_DERIVATIVE_SMOOTH_WINDOW_ALT,
            "interval_hours": POPULATION_DERIVATIVE_INTERVAL_HOURS_ALT,
            "color": "#55a868",
            "method_name": "rolling mean",
        },
        {
            "window": 41,
            "interval_hours": 8.0,
            "color": "#c44e52",
            "method_name": "rolling mean",
        },
    ]
    threshold_free_derivative_compare_specs = [
        {
            "metric_name": "organoid_mean_intensity_z",
            "title": "Whole-cyst mean intensity",
            "ylabel": "sigma per hour",
        },
        {
            "metric_name": "brightest_decile_mean_intensity_z",
            "title": "Brightest 10% mean intensity",
            "ylabel": "sigma per hour",
        },
    ]
    positive_fraction_derivative_compare_specs = [
        {
            "metric_name": "positive_fraction_sigma3",
            "title": "Positive fraction (3 sigma mask)",
            "ylabel": "fraction per hour",
        },
        {
            "metric_name": "positive_fraction_sigma4",
            "title": "Positive fraction (4 sigma mask)",
            "ylabel": "fraction per hour",
        },
    ]
    positive_mean_derivative_compare_specs = [
        {
            "metric_name": f"positive_mean_intensity_sigma{int(sigma_multiple)}_z",
            "title": f"{int(sigma_multiple)} sigma threshold",
            "ylabel": "sigma per hour",
        }
        for sigma_multiple in POPULATION_THRESHOLD_SIGMAS
    ]

    display(Markdown("### Positive-fraction threshold sweep"))
    display(
        Markdown(
            '''
            This subsection is meant to make the sigma-threshold choice physically interpretable.

            For each reporter:

            - first, the population positive-fraction traces are shown across the candidate shared within-cyst baseline `+ N sigma` thresholds
            - then, ten representative single-frame corrected images show what those same thresholds look like as cleaned positive masks in real cysts
            '''
        )
    )
    for reporter in ["RFP", "YFP"]:
        display(
            Markdown(
                f"#### {reporter}: population positive fraction across threshold choices"
            )
        )
        render_single_reporter_population_grid(
            positive_fraction_specs,
            reporter,
            FIGURE_DIR / f"05_population_positive_fraction_threshold_sweep_{reporter.lower()}.png",
            "positive fraction across threshold choices",
            yscale="linear",
            sharey=True,
        )
        display(
            Markdown(
                f"#### {reporter}: what those threshold choices look like on representative corrected images"
            )
        )
        render_threshold_choice_demo(
            reporter,
            FIGURE_DIR / f"05_population_positive_fraction_threshold_examples_{reporter.lower()}.png",
            [int(sigma_multiple) for sigma_multiple in POPULATION_THRESHOLD_SIGMAS],
        )

    display(Markdown("#### Aggregate derivative comparisons for positive fraction"))
    render_derivative_comparison_grid(
        positive_fraction_derivative_compare_specs,
        FIGURE_DIR / "05_population_positive_fraction_derivative_comparison.png",
        "Positive-fraction derivative comparison: derivative of aggregate mean vs mean of per-position derivatives",
        derivative_parameter_choices,
    )

    display(Markdown("### Threshold-free whole-cyst and brightest-pixel summaries"))
    render_population_grid(
        corrected_intensity_specs,
        FIGURE_DIR / "05_population_threshold_free_corrected_intensity.png",
        "Threshold-free reporter summaries in corrected intensity units",
        yscale="linear",
        sharey_mode=False,
        line_stat="mean",
        show_individual=True,
        show_band=False,
    )
    render_population_grid(
        corrected_symlog_specs,
        FIGURE_DIR / "05_population_threshold_free_corrected_intensity_symlog.png",
        "Threshold-free reporter summaries in corrected intensity units (symlog alternative)",
        yscale="symlog",
        sharey_mode=False,
        line_stat="mean",
        show_individual=True,
        show_band=False,
    )
    render_population_grid(
        standardized_intensity_specs,
        FIGURE_DIR / "05_population_threshold_free_standardized_intensity.png",
        "Threshold-free reporter summaries standardized to the pooled reporter null",
        yscale="linear",
        sharey_mode=False,
        line_stat="mean",
        show_individual=True,
        show_band=False,
    )
    display(Markdown("#### Corresponding derivatives for the threshold-free summaries"))
    render_derivative_comparison_grid(
        threshold_free_derivative_compare_specs,
        FIGURE_DIR / "05_population_threshold_free_derivative_comparison.png",
        "Threshold-free derivative comparison: derivative of aggregate mean vs mean of per-position derivatives",
        derivative_parameter_choices,
    )

    display(Markdown("### Threshold-based mean intensity within the positive region"))
    render_population_grid(
        positive_mean_specs,
        FIGURE_DIR / "05_population_positive_mean_intensity_threshold_sweep.png",
        "Threshold-based mean reporter intensity within the positive region",
        yscale="linear",
        sharey_mode=False,
        line_stat="mean",
        show_individual=True,
        show_band=False,
    )
    render_population_grid(
        positive_mean_standardized_specs,
        FIGURE_DIR / "05_population_positive_mean_intensity_threshold_sweep_standardized.png",
        "Threshold-based mean reporter intensity within the positive region, standardized to the pooled reporter null",
        yscale="linear",
        sharey_mode=False,
        line_stat="mean",
        show_individual=True,
        show_band=False,
    )
    display(Markdown("#### Corresponding derivatives for the positive-region mean intensity"))
    render_derivative_comparison_grid(
        positive_mean_derivative_compare_specs,
        FIGURE_DIR / "05_population_positive_mean_intensity_derivative_comparison.png",
        "Positive-region mean-intensity derivative comparison: derivative of aggregate mean vs mean of per-position derivatives",
        derivative_parameter_choices,
    )

    display(Markdown("### Aggregate-mean derivative estimator comparisons"))
    derivative_exploration_specs = [
        {
            "metric_name": "organoid_mean_intensity_z",
            "title": "Whole-cyst mean intensity",
            "ylabel": "sigma per hour",
        },
        {
            "metric_name": "positive_mean_intensity_sigma4_z",
            "title": "Positive-region mean intensity\n(4 sigma mask)",
            "ylabel": "sigma per hour",
        },
        {
            "metric_name": "positive_fraction_sigma4",
            "title": "Positive fraction\n(4 sigma mask)",
            "ylabel": "fraction per hour",
        },
    ]
    render_aggregate_derivative_window_sweep(
        derivative_exploration_specs,
        FIGURE_DIR / "05_population_derivative_window_sweep.png",
        "Derivative exploration from aggregate mean traces: rolling-mean window sweep",
    )
    render_aggregate_derivative_interval_sweep(
        derivative_exploration_specs,
        FIGURE_DIR / "05_population_derivative_interval_sweep.png",
        "Derivative exploration from aggregate mean traces: centered-slope interval sweep",
    )
    render_aggregate_derivative_method_sweep(
        derivative_exploration_specs,
        FIGURE_DIR / "05_population_derivative_method_sweep.png",
        "Derivative exploration from aggregate mean traces: smoothing-method comparison",
    )


## Review Trace Outlier QC

This is a lightweight watch-list for unusual per-position reporter traces.

The goal is not to decide biology from a table. The goal is to identify positions with unusually large single-frame changes, then carry those positions forward into the integrated per-position review so the images and traces can be judged together.


In [ ]:
if DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING:
    trace_qc_df = pd.DataFrame()
    flagged_trace_df = pd.DataFrame()
    flagged_trace_positions = []
    display(Markdown("Skipping trace-outlier QC because `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = True`."))
else:
    trace_qc_rows = []
    if population_trace_metrics_df.empty or not trace_qc_metric_specs:
        trace_qc_df = pd.DataFrame()
        flagged_trace_df = pd.DataFrame()
        flagged_trace_positions = []
        display(Markdown("No trace-outlier QC values could be computed from the current population-trace metrics."))
    else:
        metric_title_map = {
            spec["metric_name"]: spec["metric_title"]
            for spec in trace_qc_metric_specs
        }
        metric_short_map = {
            spec["metric_name"]: spec["metric_short"]
            for spec in trace_qc_metric_specs
        }

        for reporter in REPORTER_COLORS:
            subset = population_trace_metrics_df.loc[
                population_trace_metrics_df["reporter"] == reporter
            ].sort_values(["position_label", "time_index"])
            for position_label, group in subset.groupby("position_label", sort=True):
                time_indices = group["time_index"].astype(int).to_numpy()
                time_hours = group["time_hours"].astype(float).to_numpy()
                for spec in trace_qc_metric_specs:
                    metric_name = str(spec["metric_name"])
                    metric_title = metric_title_map[metric_name]
                    values = group[metric_name].astype(float).to_numpy()
                    finite_mask = np.isfinite(values)
                    if finite_mask.sum() < 2:
                        continue
                    values = values[finite_mask]
                    metric_time_indices = time_indices[finite_mask]
                    metric_time_hours = time_hours[finite_mask]
                    diffs = np.abs(np.diff(values))
                    if diffs.size == 0 or not np.isfinite(diffs).any():
                        continue
                    max_step_idx = int(np.nanargmax(diffs))
                    max_step_value = float(np.nanmax(diffs))
                    median_step_value = float(np.nanmedian(diffs))
                    trace_qc_rows.append(
                        {
                            "position_label": position_label,
                            "reporter": reporter,
                            "metric_name": metric_name,
                            "metric_title": metric_title,
                            "metric_short": metric_short_map[metric_name],
                            "max_abs_step": max_step_value,
                            "median_abs_step": median_step_value,
                            "step_ratio": float(max_step_value / max(median_step_value, 1e-6)),
                            "max_step_time_prev": int(metric_time_indices[max_step_idx]),
                            "max_step_time_curr": int(metric_time_indices[max_step_idx + 1]),
                            "max_step_hour_prev": float(metric_time_hours[max_step_idx]),
                            "max_step_hour_curr": float(metric_time_hours[max_step_idx + 1]),
                        }
                    )

        trace_qc_df = pd.DataFrame(trace_qc_rows)
        if trace_qc_df.empty:
            flagged_trace_df = pd.DataFrame()
            flagged_trace_positions = []
            display(Markdown("No trace-outlier QC values could be computed from the current population-trace metrics."))
        else:
            trace_qc_df["max_step_threshold"] = trace_qc_df.groupby(["reporter", "metric_name"])["max_abs_step"].transform(
                lambda s: float(s.quantile(0.95))
            )
            trace_qc_df["flagged"] = trace_qc_df["max_abs_step"] >= trace_qc_df["max_step_threshold"]
            trace_qc_df["severity_score"] = (
                trace_qc_df["max_abs_step"] / trace_qc_df["max_step_threshold"].replace(0, np.nan)
            ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
            flagged_trace_df = trace_qc_df.loc[trace_qc_df["flagged"]].copy()
            flagged_trace_df["flag_window_t"] = flagged_trace_df.apply(
                lambda row: f"t={format_display_hours_from_index(row['max_step_time_prev'], 2)}->{format_display_hours_from_index(row['max_step_time_curr'], 2)}",
                axis=1,
            )
            flagged_trace_df["flag_window_h"] = flagged_trace_df.apply(
                lambda row: f"{display_time_hours(float(row['max_step_hour_prev'])):.2f}->{display_time_hours(float(row['max_step_hour_curr'])):.2f} h",
                axis=1,
            )
            flagged_trace_df["flag_detail"] = flagged_trace_df.apply(
                lambda row: (
                    f"{row['reporter']} {row['metric_short']} jump "
                    f"at t={format_display_hours_from_index(row['max_step_time_prev'], 2)}->{format_display_hours_from_index(row['max_step_time_curr'], 2)} "
                    f"({display_time_hours(float(row['max_step_hour_prev'])):.2f}->{display_time_hours(float(row['max_step_hour_curr'])):.2f} h); "
                    f"step={float(row['max_abs_step']):.3g}, threshold={float(row['max_step_threshold']):.3g}, "
                    f"ratio={float(row['step_ratio']):.2f}x"
                ),
                axis=1,
            )
            flagged_trace_df = flagged_trace_df.sort_values(
                ["severity_score", "reporter", "metric_name", "position_label"],
                ascending=[False, True, True, True],
            ).reset_index(drop=True)
            flagged_trace_positions = flagged_trace_df["position_label"].drop_duplicates().tolist()

            trace_outlier_table_path = TABLE_DIR / "05_trace_outliers.tsv"
            flagged_trace_df.to_csv(trace_outlier_table_path, sep="\t", index=False)

            display(
                Markdown(
                    f'''
                    **Trace outlier watch-list**

                    - flagged reporter-position-metric traces: `{len(flagged_trace_df)}`
                    - flagged positions (union across metrics/reporters): `{len(flagged_trace_positions)}`
                    - source metrics: threshold-free standardized cyst summaries plus positive-fraction traces at `3 sigma` and `4 sigma`
                    - criterion: largest single-frame step above the `95th percentile` within each reporter and trace metric
                    - each flagged row below tells you exactly **which reporter/metric** triggered the flag and **between which two frames** the jump occurred

                    Wrote flagged trace table: `{trace_outlier_table_path.name}`
                    '''
                )
            )
            if flagged_trace_df.empty:
                display(Markdown("No trace outliers were flagged by the current rules."))
            else:
                display(
                    flagged_trace_df[
                        [
                            "position_label",
                            "reporter",
                            "metric_short",
                            "max_abs_step",
                            "median_abs_step",
                            "step_ratio",
                            "max_step_threshold",
                            "flag_window_t",
                            "flag_window_h",
                            "flag_detail",
                        ]
                    ]
                )


def select_trace_problem_times(summary_row: pd.Series, time_indices: list[int]) -> list[int]:
    if not time_indices:
        return []
    prev_time = int(summary_row["max_step_time_prev"])
    curr_time = int(summary_row["max_step_time_curr"])
    if prev_time in time_indices:
        prev_idx = time_indices.index(prev_time)
    else:
        prev_idx = max(0, len(time_indices) // 2 - 1)
    candidate_indices = [max(0, prev_idx - 1), prev_idx, min(len(time_indices) - 1, prev_idx + 1), min(len(time_indices) - 1, prev_idx + 2)]
    selected = []
    for idx in candidate_indices:
        time_index = int(time_indices[idx])
        if time_index not in selected:
            selected.append(time_index)
    return selected


def assemble_review_times(
    available_times: list[int],
    preferred_times: list[int],
    target_count: int = 4,
) -> list[int]:
    if not available_times:
        return []
    ordered_preferred = []
    available_set = set(int(time_index) for time_index in available_times)
    for time_index in preferred_times:
        time_index = int(time_index)
        if time_index in available_set and time_index not in ordered_preferred:
            ordered_preferred.append(time_index)
    evenly_spaced = [int(available_times[idx]) for idx in np.linspace(0, len(available_times) - 1, target_count).round().astype(int)]
    for time_index in evenly_spaced:
        if time_index not in ordered_preferred:
            ordered_preferred.append(time_index)
    ordered_preferred = sorted(ordered_preferred, key=lambda time_index: available_times.index(time_index))
    if len(ordered_preferred) <= target_count:
        return ordered_preferred
    chosen = []
    for idx in np.linspace(0, len(ordered_preferred) - 1, target_count).round().astype(int):
        time_index = int(ordered_preferred[idx])
        if time_index not in chosen:
            chosen.append(time_index)
    for time_index in ordered_preferred:
        if len(chosen) >= target_count:
            break
        if time_index not in chosen:
            chosen.append(time_index)
    return chosen[:target_count]


## Integrated Position Review

This is the main per-position inspection block.

For each review position, the notebook now keeps the relevant pieces together:

- representative phase / reporter images
- the current reporter-positive masks
- the whole off-cyst background trace
- the reporter traces used downstream

That way, a position can be judged in one place instead of forcing image review, QC review, and trace review into separate sections.


In [ ]:
if DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING:
    integrated_review_positions = []
    integrated_review_times = {}
    integrated_review_reasons = {}
    integrated_review_background_flags = {}
    integrated_review_trace_flags = {}
    overlay_corrected_display_limits = {}
    display(Markdown("Skipping integrated position review selection because `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = True`."))
else:
    flagged_position_labels = flagged_position_qc["position_label"].tolist() if "flagged_position_qc" in globals() and not flagged_position_qc.empty else []
    flagged_trace_position_labels = flagged_trace_df["position_label"].drop_duplicates().tolist() if "flagged_trace_df" in globals() and not flagged_trace_df.empty else []
    integrated_review_positions = flagged_trace_position_labels if flagged_trace_position_labels else flagged_position_labels

    integrated_review_reasons = {}
    integrated_review_times = {}
    integrated_review_background_flags = {}
    integrated_review_trace_flags = {}
    overlay_corrected_display_limits = {}

    for position_label in integrated_review_positions:
        reason_parts = []
        flag_rows = (
            flagged_background_df.loc[flagged_background_df["position_label"] == position_label]
            .sort_values(["severity_score", "reporter"], ascending=[False, True])
            if "flagged_background_df" in globals() and not flagged_background_df.empty
            else pd.DataFrame()
        )
        trace_rows = (
            flagged_trace_df.loc[flagged_trace_df["position_label"] == position_label]
            .sort_values(["severity_score", "reporter", "metric_name"], ascending=[False, True, True])
            if "flagged_trace_df" in globals() and not flagged_trace_df.empty
            else pd.DataFrame()
        )
        integrated_review_background_flags[position_label] = flag_rows.copy()
        integrated_review_trace_flags[position_label] = trace_rows.copy()
        if not flag_rows.empty:
            top_flag = flag_rows.iloc[0]
            background_label = top_flag["issue_label"] if "issue_label" in top_flag and pd.notna(top_flag["issue_label"]) and str(top_flag["issue_label"]).strip() else top_flag["issue_class"]
            reason_parts.append(f"background QC: {top_flag['reporter']} {background_label}")
        if not trace_rows.empty:
            top_trace = trace_rows.iloc[0]
            reason_parts.append(
                f"trace QC: {top_trace['reporter']} {top_trace['metric_short']} jump {top_trace['flag_window_t']}"
            )
        integrated_review_reasons[position_label] = "; ".join(reason_parts) if reason_parts else "review"

        subset = frame_metrics.loc[
            (frame_metrics["position_label"] == position_label)
            & (~frame_metrics["exclude_from_analysis"])
            & (frame_metrics["reporter"] == "RFP")
        ].sort_values("time_index")
        available_times = subset["time_index"].astype(int).tolist()
        if not available_times:
            integrated_review_times[position_label] = []
            continue

        preferred_times = []
        if not flag_rows.empty:
            preferred_times.extend(select_background_problem_times(flag_rows.iloc[0], available_times))
        if not trace_rows.empty:
            preferred_times.extend(select_trace_problem_times(trace_rows.iloc[0], available_times))
        frame_times = assemble_review_times(
            available_times,
            preferred_times,
            target_count=max(4, DEBUG_INTEGRATED_REVIEW_TIMEPOINT_COUNT),
        )

        integrated_review_times[position_label] = frame_times
        overlay_corrected_display_limits[position_label] = {}

        if not frame_times:
            continue
        early_time_index = int(frame_times[0])
        for reporter in ["RFP", "YFP"]:
            reporter_image = load_reporter_after_illumination(position_label, reporter, early_time_index).astype(float)
            reporter_row = frame_metrics.loc[
                (frame_metrics["position_label"] == position_label)
                & (frame_metrics["time_index"] == early_time_index)
                & (frame_metrics["reporter"] == reporter)
            ].iloc[0]
            corrected = reporter_image - float(reporter_row["background_value"])
            overlay_corrected_display_limits[position_label][reporter] = corrected_display_limits_from_arrays([corrected])

    review_manifest = pd.DataFrame(
        [
            {
                "position_label": position_label,
                "background_flag": (
                    f"{integrated_review_background_flags[position_label].iloc[0]['reporter']} "
                    f"{integrated_review_background_flags[position_label].iloc[0]['issue_label'] if pd.notna(integrated_review_background_flags[position_label].iloc[0].get('issue_label', np.nan)) and str(integrated_review_background_flags[position_label].iloc[0].get('issue_label', '')).strip() else integrated_review_background_flags[position_label].iloc[0]['issue_class']}"
                    if position_label in integrated_review_background_flags and not integrated_review_background_flags[position_label].empty
                    else ""
                ),
                "background_flag_detail": (
                    integrated_review_background_flags[position_label].iloc[0]["issue_detail"]
                    if position_label in integrated_review_background_flags
                    and not integrated_review_background_flags[position_label].empty
                    and "issue_detail" in integrated_review_background_flags[position_label].columns
                    else ""
                ),
                "trace_flag": (
                    f"{integrated_review_trace_flags[position_label].iloc[0]['reporter']} "
                    f"{integrated_review_trace_flags[position_label].iloc[0]['metric_short']} jump "
                    f"{integrated_review_trace_flags[position_label].iloc[0]['flag_window_t']}"
                    if position_label in integrated_review_trace_flags and not integrated_review_trace_flags[position_label].empty
                    else ""
                ),
                "trace_flag_detail": (
                    integrated_review_trace_flags[position_label].iloc[0]["flag_detail"]
                    if position_label in integrated_review_trace_flags and not integrated_review_trace_flags[position_label].empty
                    else ""
                ),
                "reason": integrated_review_reasons[position_label],
                "review_times": ", ".join(
                    format_display_hours_from_index(time_index, 2)
                    for time_index in integrated_review_times.get(position_label, [])
                ),
            }
            for position_label in integrated_review_positions
        ]
    )
    review_manifest_path = TABLE_DIR / "05_integrated_review_selection.tsv"
    display_time_df(review_manifest).to_csv(review_manifest_path, sep="\t", index=False)
    display(
        Markdown(
            "### Review Positions\n\n`" + ", ".join(integrated_review_positions) + "`"
        )
    )
    display(Markdown(f"**Integrated review selection manifest**  \n`{review_manifest_path.name}`"))
    display(display_time_df(review_manifest))
    print("Wrote integrated review selection table:", review_manifest_path)


In [ ]:
if DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING:
    display(Markdown("Skipping integrated position review figures because `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = True`."))
else:
    for position_label in integrated_review_positions:
        frame_times = integrated_review_times.get(position_label, [])
        if not frame_times:
            continue

        background_flag_rows = integrated_review_background_flags.get(position_label, pd.DataFrame())
        trace_flag_rows = integrated_review_trace_flags.get(position_label, pd.DataFrame())
        low, high = position_display_limits(position_label)
        total_cols = max(len(frame_times), 4)
        fig_width = max(16.5, 3.2 * total_cols)
        fig = plt.figure(figsize=(fig_width, 12.0), constrained_layout=True)
        gs = fig.add_gridspec(4, total_cols, height_ratios=[1.0, 1.0, 1.0, 1.15])
        review_time_hours = []
        time_hour_lookup = {}
        for time_index in frame_times:
            time_hours = float(
                frame_metrics.loc[
                    (frame_metrics["position_label"] == position_label)
                    & (frame_metrics["reporter"] == "RFP")
                    & (frame_metrics["time_index"] == time_index),
                    "time_hours",
                ].iloc[0]
            )
            review_time_hours.append(time_hours)
            time_hour_lookup[int(time_index)] = time_hours

        display(
            Markdown(
                f'''
                ### {position_label}

                - review reason: `{integrated_review_reasons.get(position_label, 'review')}`
                - review times: `{", ".join(format_display_hours_from_index(time_index, 2) for time_index in frame_times)}`
                '''
            )
        )

        for col_index, time_index in enumerate(frame_times):
            phase = load_phase(position_label, time_index)
            mask = load_mask(position_label, time_index)
            phase_disp = np.clip((phase.astype(float) - low) / (high - low), 0, 1)

            ax_phase = fig.add_subplot(gs[0, col_index])
            ax_phase.imshow(phase_disp, cmap="gray")
            draw_mask(ax_phase, mask, color="deepskyblue", linewidth=2.0)
            ax_phase.set_title(f"t={format_display_hours_from_index(time_index, 2)}", fontsize=10.5, pad=10)
            if col_index == 0:
                ax_phase.text(-0.08, 0.5, "Phase", transform=ax_phase.transAxes, ha="right", va="center", fontsize=10, fontweight="bold")
            ax_phase.axis("off")

            for row_index, reporter in enumerate(["RFP", "YFP"], start=1):
                reporter_image = load_reporter_after_illumination(position_label, reporter, time_index).astype(float)
                reporter_row = frame_metrics.loc[
                    (frame_metrics["position_label"] == position_label)
                    & (frame_metrics["time_index"] == time_index)
                    & (frame_metrics["reporter"] == reporter)
                ].iloc[0]
                corrected = reporter_image - float(reporter_row["background_value"])
                positive_mask = positive_mask_from_corrected(
                    corrected=corrected,
                    organoid_mask=mask,
                    threshold_value=float(reporter_row["threshold_value"]),
                )
                display_vmin, display_vmax = overlay_corrected_display_limits[position_label][reporter]

                ax = fig.add_subplot(gs[row_index, col_index])
                ax.imshow(
                    np.clip(corrected, display_vmin, display_vmax),
                    cmap="magma",
                    vmin=display_vmin,
                    vmax=display_vmax,
                )
                draw_mask(ax, mask, color="deepskyblue", linewidth=1.8)
                draw_mask(ax, positive_mask, color="chartreuse", linewidth=1.8)
                if col_index == 0:
                    ax.text(-0.08, 0.5, reporter, transform=ax.transAxes, ha="right", va="center", fontsize=10, fontweight="bold")
                ax.text(
                    0.03,
                    0.04,
                    f"bg={reporter_row['background_value']:.0f} | frac(4sigma)={reporter_row['positive_fraction_raw']:.3f}",
                    transform=ax.transAxes,
                    ha="left",
                    va="bottom",
                    fontsize=7.9,
                    color="white",
                    bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0),
                )
                ax.axis("off")

        bottom_gs = gs[3, :].subgridspec(1, 4, wspace=0.3)

        ax_bg = fig.add_subplot(bottom_gs[0, 0])
        for reporter, color in REPORTER_COLORS.items():
            trace_df = whole_background_trace_df(position_label, reporter)
            ax_bg.plot(trace_df["time_hours"], trace_df["background_value"], color=color, linewidth=2.0, label=f"{reporter} whole bg")
        if not background_flag_rows.empty:
            top_bg = background_flag_rows.iloc[0]
            prev_hour = time_hour_lookup.get(int(top_bg["max_step_time_prev"]))
            curr_hour = time_hour_lookup.get(int(top_bg["max_step_time_curr"]))
            if prev_hour is None or curr_hour is None:
                rep_subset = frame_metrics.loc[
                    (frame_metrics["position_label"] == position_label)
                    & (frame_metrics["reporter"] == "RFP")
                    & (~frame_metrics["exclude_from_analysis"])
                ].sort_values("time_index")
                hour_map = rep_subset.set_index("time_index")["time_hours"].to_dict()
                prev_hour = hour_map.get(int(top_bg["max_step_time_prev"]))
                curr_hour = hour_map.get(int(top_bg["max_step_time_curr"]))
            if prev_hour is not None and curr_hour is not None:
                ax_bg.axvspan(prev_hour, curr_hour, color="0.85", alpha=0.6, zorder=0)
        for time_hours in review_time_hours:
            ax_bg.axvline(time_hours, color="0.35", linestyle=":", linewidth=1.0)
        ax_bg.set_title("Whole off-cyst background")
        ax_bg.set_xlabel("Time (hours)")
        set_display_time_axis(ax_bg, "x")
        ax_bg.set_ylabel("Background")
        ax_bg.legend(loc="best", fontsize=8)
        ax_bg.grid(alpha=0.2)

        subset = population_trace_metrics_df.loc[
            population_trace_metrics_df["position_label"] == position_label
        ].sort_values("time_index")
        for col_index, (metric_name, metric_title) in enumerate(TRACE_METRICS, start=1):
            ax = fig.add_subplot(bottom_gs[0, col_index])
            for reporter, color in REPORTER_COLORS.items():
                rep = subset.loc[subset["reporter"] == reporter]
                ax.plot(
                    rep["time_hours"],
                    rep[metric_name],
                    color=color,
                    linewidth=2.0,
                    linestyle=REPORTER_LINESTYLES[reporter],
                    label=reporter,
                )
            metric_trace_flags = trace_flag_rows.loc[
                trace_flag_rows["metric_name"] == metric_name
            ] if not trace_flag_rows.empty else pd.DataFrame()
            for _, trace_flag_row in metric_trace_flags.iterrows():
                ax.axvspan(
                    float(trace_flag_row["max_step_hour_prev"]),
                    float(trace_flag_row["max_step_hour_curr"]),
                    color="0.85",
                    alpha=0.6,
                    zorder=0,
                )
            for time_hours in review_time_hours:
                ax.axvline(time_hours, color="0.35", linestyle=":", linewidth=1.0)
            if not metric_trace_flags.empty:
                trace_notes = [
                    (
                        f"{row['reporter']} {row['flag_window_t']} "
                        f"(step={float(row['max_abs_step']):.3g})"
                    )
                    for _, row in metric_trace_flags.iterrows()
                ]
                ax.text(
                    0.02,
                    0.98,
                    "Flagged: " + " | ".join(trace_notes),
                    transform=ax.transAxes,
                    ha="left",
                    va="top",
                    fontsize=7.5,
                    color="black",
                    bbox=dict(facecolor="white", alpha=0.75, edgecolor="0.7", pad=2.0),
                )
            if metric_name.endswith("_z") or metric_name.endswith("_dt"):
                ax.axhline(0, color="0.55", linestyle=":", linewidth=1.0, zorder=0)
            ax.set_title(metric_title)
            ax.set_xlabel("Time (hours)")
            set_display_time_axis(ax, "x")
            if col_index == 1:
                ax.legend(loc="best", fontsize=8)
            ax.grid(alpha=0.2)

        fig.suptitle(
            (
                f"{position_label} integrated review | {integrated_review_reasons.get(position_label, 'review')} | "
                f"RFP display [{overlay_corrected_display_limits[position_label]['RFP'][0]:.0f}, {overlay_corrected_display_limits[position_label]['RFP'][1]:.0f}] | "
                f"YFP display [{overlay_corrected_display_limits[position_label]['YFP'][0]:.0f}, {overlay_corrected_display_limits[position_label]['YFP'][1]:.0f}]"
            ),
            fontsize=12,
            y=1.01,
        )

        overlay_path = PREVIEW_DIR / f"{position_label}_integrated_review.png"
        fig.savefig(overlay_path, dpi=180, bbox_inches="tight")
        display(fig)
        plt.close(fig)

    print("Wrote integrated review previews to:", PREVIEW_DIR)


## Review Preliminary Onset Calls

These onset calls are provisional. They are useful now as a quick read on whether the aggregate traces are behaving in the expected RFP-before-YFP direction.

This section is skipped when `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = True`.


In [ ]:
if DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING:
    display(Markdown("Skipping preliminary onset plots because `DEBUG_SKIP_DOWNSTREAM_IMAGE_PLOTTING = True`."))
else:
    onset_plot_df = onset_summary.dropna(subset=["rfp_onset_time_hours", "yfp_onset_time_hours"]).copy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    if not onset_plot_df.empty:
        xy_min = float(np.nanmin(onset_plot_df[["rfp_onset_time_hours", "yfp_onset_time_hours"]].to_numpy()))
        xy_max = float(np.nanmax(onset_plot_df[["rfp_onset_time_hours", "yfp_onset_time_hours"]].to_numpy()))
        axes[0].scatter(
            onset_plot_df["rfp_onset_time_hours"],
            onset_plot_df["yfp_onset_time_hours"],
            color="black",
            alpha=0.8,
        )
        axes[0].plot([xy_min, xy_max], [xy_min, xy_max], linestyle="--", color="gray")
        axes[0].set_xlabel("RFP onset (hours)")
        axes[0].set_ylabel("YFP onset (hours)")
        set_display_time_axis(axes[0], "both", crowded=True)
        axes[0].set_title("Per-position preliminary onset times")

        axes[1].hist(onset_plot_df["yfp_minus_rfp_onset_hours"].dropna(), bins=12, color="steelblue", edgecolor="black")
        axes[1].axvline(0, linestyle="--", color="gray")
        axes[1].set_xlabel("YFP onset - RFP onset (hours)")
        axes[1].set_ylabel("Position count")
        axes[1].set_title("Preliminary onset lag distribution")
    else:
        axes[0].text(0.5, 0.5, "No positions with both onset calls", ha="center", va="center")
        axes[1].text(0.5, 0.5, "No positions with both onset calls", ha="center", va="center")
        for ax in axes:
            ax.set_axis_off()

    onset_fig_path = FIGURE_DIR / "05_preliminary_onset_summary.png"
    fig.savefig(onset_fig_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote preliminary onset summary:", onset_fig_path)


## Save Stage Outputs

The quantification tables are written by the standalone script. This final cell reports the main figure and preview outputs generated by the notebook.


In [ ]:
created_paths = [
    FIGURE_DIR / "05_reporter_threshold_summary.png",
    FIGURE_DIR / "05_population_positive_fraction_threshold_sweep_rfp.png",
    FIGURE_DIR / "05_population_positive_fraction_threshold_examples_rfp.png",
    FIGURE_DIR / "05_population_positive_fraction_threshold_sweep_yfp.png",
    FIGURE_DIR / "05_population_positive_fraction_threshold_examples_yfp.png",
    FIGURE_DIR / "05_population_threshold_free_corrected_intensity.png",
    FIGURE_DIR / "05_population_threshold_free_standardized_intensity.png",
    FIGURE_DIR / "05_population_threshold_free_derivative_comparison.png",
    FIGURE_DIR / "05_population_positive_fraction_derivative_comparison.png",
    FIGURE_DIR / "05_population_positive_mean_intensity_threshold_sweep.png",
    FIGURE_DIR / "05_population_positive_mean_intensity_threshold_sweep_standardized.png",
    FIGURE_DIR / "05_population_positive_mean_intensity_derivative_comparison.png",
    FIGURE_DIR / "05_population_derivative_method_sweep.png",
    FIGURE_DIR / "05_population_derivative_window_sweep.png",
    FIGURE_DIR / "05_population_derivative_interval_sweep.png",
    FIGURE_DIR / "05_preliminary_onset_summary.png",
]

print("Created files")
for path in created_paths:
    if path.exists():
        print("-", path.relative_to(ROOT).as_posix())

print("\nPreview directory")
print("-", PREVIEW_DIR.relative_to(ROOT).as_posix())


## Reading The Outputs

The most useful things to inspect after this notebook runs are:

- `results/previews/05_preliminary_reporter_quantification/`
- `results/figures/05/05_population_positive_fraction_threshold_sweep_rfp.png`
- `results/figures/05/05_population_positive_fraction_threshold_examples_rfp.png`
- `results/figures/05/05_population_positive_fraction_threshold_sweep_yfp.png`
- `results/figures/05/05_population_positive_fraction_threshold_examples_yfp.png`
- `results/figures/05/05_population_threshold_free_corrected_intensity.png`
- `results/figures/05/05_population_threshold_free_standardized_intensity.png`
- `results/figures/05/05_population_threshold_free_derivative_comparison.png`
- `results/figures/05/05_population_positive_fraction_derivative_comparison.png`
- `results/figures/05/05_population_positive_mean_intensity_threshold_sweep.png`
- `results/figures/05/05_population_positive_mean_intensity_threshold_sweep_standardized.png`
- `results/figures/05/05_population_positive_mean_intensity_derivative_comparison.png`
- `results/figures/05/05_population_derivative_method_sweep.png`
- `results/figures/05/05_population_derivative_window_sweep.png`
- `results/figures/05/05_population_derivative_interval_sweep.png`
- `results/figures/05/05_preliminary_onset_summary.png`

The next practical decision after this notebook is whether the current reporter thresholding is good enough to carry into manuscript-style traces, or whether one more small refinement pass is worth doing before figure assembly.
